# MIMOSA figures reworked

Consolidated publication-figure analysis for the MIMOSA creep study.

This is the reworked copy of ``figures.py``.  The original file is intentionally
unchanged.  The module is organized as executable ``# %%`` cells, but is also
valid Python and can be imported for tests.

There is one sectioning operator, ``manuscript_windows_256x128``.  The fixed
experimental analysis crop ``[50:-50, 50:750]`` is resampled to a 1 um
pixel-centre *analysis* grid and split into the largest centred grid of
non-overlapping 256-by-128 um windows (21 windows for a standard cropped 10x
map).  Each window is independently plane-levelled.  Simulation boundary
planes use the same physical footprint.  Interpolation does not improve the
native 10x optical resolution or raise its Nyquist limit.

The pre-existing non-sectioned work remains available on the whole analysis
crop and does not introduce a competing section grid.  The uncropped native
map is retained only for acquisition/QC display.

Only the four raw-data caches below are used by downstream cells:
``exp_df``, ``sim_df``, ``exp_heights``, and ``sim_heights``.  Raw experimental
CSVs and ``SimResults`` objects are read only when those caches are rebuilt.


## Imports, shared style, and configuration


In [ ]:
from __future__ import annotations

from contextlib import AbstractContextManager
from dataclasses import dataclass
from functools import lru_cache
from io import StringIO
from pathlib import Path
from typing import Iterable, Iterator, Mapping, Sequence
import hashlib
import json
import warnings
import zipfile

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import binary_dilation, distance_transform_edt, gaussian_filter, label
from scipy.optimize import curve_fit
from scipy.signal import fftconvolve

from utils.config import (
    DATA_DIR,
    MICROSTRUCTURE_DIR,
    PROFILOMETRY_SPACING_UM,
    RC_PARAMS,
    RESULTS_DIR,
    VOXELSIZE,
)
from utils.data_utils import SimResults


# RC_PARAMS is the only global Matplotlib style source in this file.
plt.rcParams.update(RC_PARAMS)

PROJECT_ROOT = Path(DATA_DIR).resolve().parent
OUTPUT_DIR = Path(RESULTS_DIR) / "figures_reworked"
CACHE_DIR = Path(RESULTS_DIR) / "figure_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_SCHEMA_VERSION = 3
POLISH = "polished"
MAGNIFICATION = "10x"

EXPERIMENTS = (
    (475, "int"),
    (500, "unint"),
    (525, "int"),
    (530, "unint"),
    (575, "int"),
    (588, "unint"),
)
INTERRUPTED_CASES = ((475, "int"), (525, "int"), (575, "int"))
UNINTERRUPTED_CASES = ((500, "unint"), (530, "unint"), (588, "unint"))

# One color per experiment, used for both experimental and simulated data.
LOAD_COLORS = {
    475: "tab:blue",
    500: "tab:orange",
    525: "tab:green",
    530: "tab:red",
    575: "tab:purple",
    588: "tab:brown",
}
SOURCE_LINESTYLES = {"exp": "-", "sim": "--"}
TYPE_MARKERS = {"int": "o", "unint": "s"}

# These are independent quantities.  Do not use a PSD wavelength as an ACF lag.
WAVELENGTH_MIN_UM = 4.1264
WAVELENGTH_MAX_UM = 60.8714
ACF_MAX_LAG_UM = 64.0

# Figure 10 / matched morphology operator.
TARGET_SPACING_UM = 1.0
SPATIAL_WINDOW_SHAPE_UM = (256.0, 128.0)
N_MATCHED_FREQUENCY_ANNULI = 17
ACF_RADIAL_BIN_WIDTH_UM = 1.0
EXP_NATIVE_SPACING_UM = float(PROFILOMETRY_SPACING_UM[MAGNIFICATION])
EXP_NATIVE_NYQUIST_UM_INV = 0.5 / EXP_NATIVE_SPACING_UM

# DIC coordinates are recorded in millimeters.  This is a distinct measurement
# bandwidth, not a second surface-height sectioning operator.
DIC_WAVELENGTH_MIN_MM = 0.8
DIC_WAVELENGTH_MAX_MM = 10.0

# Every experimental height statistic starts from this crop.  The raw native
# map remains in exp_heights so acquisition validity and QC can be audited.
EXP_ANALYSIS_CROP = (slice(50, -50), slice(50, 750))

# The four SimResults faces are ordered x-min, x-max, y-min, y-max in the
# existing extraction code.  x-max and y-max are periodic duplicates.
SIM_UNIQUE_FACE_INDICES = (0, 2)
PRIMARY_MORPHOLOGY_MICRO_IDS = ("micro1", "micro2", "micro3")

MICRO_RUNS = tuple(
    {
        "micro_id": f"micro{i}",
        "sim_root": PROJECT_ROOT / "hpc_downloads" / "gtdebru" / f"micro{i}_production",
        "microstructure": Path(MICROSTRUCTURE_DIR) / "production" / f"micro{i}_production.dat",
    }
    for i in (1, 2, 3)
)

EXP_DF_CACHE = CACHE_DIR / "exp_df.pkl"
SIM_DF_CACHE = CACHE_DIR / "sim_df.pkl"
EXP_HEIGHTS_CACHE = CACHE_DIR / "exp_heights.npz"
SIM_HEIGHTS_CACHE = CACHE_DIR / "sim_heights.npz"
CACHE_MANIFEST = CACHE_DIR / "cache_manifest.json"

CALIBRATION_PARAMS_PATH = PROJECT_ROOT / "params" / "best_row.csv"
DEFAULT_DIC_CSV_PATH: Path | None = None

# Used only for non-sectioned, scale-resolved summaries.  Bounds are clipped to
# WAVELENGTH_MIN_UM and WAVELENGTH_MAX_UM by ``validated_wavelength_bands``.
DEFAULT_WAVELENGTH_BANDS_UM = {
    "short": (WAVELENGTH_MIN_UM, 12.0),
    "intermediate": (12.0, 32.0),
    "long": (32.0, WAVELENGTH_MAX_UM),
}

EXPECTED_EXP_SHAPE = (768, 1024)
MAX_MISSING_FRACTION = 1.0e-4
MAX_MISSING_COMPONENT_PIXELS = 1
EXCLUDED_DIRECTORY_NAMES = {"bad"}


@dataclass(frozen=True)
class PSDResult:
    """One radially reduced two-dimensional periodogram."""

    frequency_um_inv: np.ndarray
    wavelength_um: np.ndarray
    wavelength_lower_um: np.ndarray
    wavelength_upper_um: np.ndarray
    wavelength_width_um: np.ndarray
    radial_mean_um4: np.ndarray
    annular_power_um2: np.ndarray
    normalized_wavelength_density_um_inv: np.ndarray
    legacy_frequency_area_normalized_um: np.ndarray
    modes: np.ndarray
    parseval_relative_error: float
    spectral_median_wavelength_um: float
    long_wavelength_power_fraction: float
    high_frequency_exponent: float


@dataclass(frozen=True)
class ACFResult:
    """Radial linear, overlap-corrected autocorrelation and landmarks."""

    lag_um: np.ndarray
    correlation: np.ndarray
    counts: np.ndarray
    one_over_e_um: float
    first_zero_um: float
    one_over_e_crossed: bool
    first_zero_crossed: bool


def load_color(load_mpa: int | float) -> str:
    """Return the fixed experiment color for a load."""
    return LOAD_COLORS[int(load_mpa)]


def strain_group_colors(n_groups: int) -> np.ndarray:
    """Return the one permitted strain-group palette (viridis-like)."""
    if n_groups < 1:
        return np.empty((0, 4))
    return mpl.colormaps["viridis"](np.linspace(0.16, 0.90, n_groups))




## Core height-map and surface calculations


In [ ]:
def trapezoid(y: np.ndarray, x: np.ndarray) -> float:
    """Compatibility wrapper around NumPy trapezoidal integration."""
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x=x))
    return float(np.trapz(y, x=x))


def finite_median(values: np.ndarray) -> float:
    """Median of finite values, or NaN when a descriptor never resolved."""
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    return float(np.median(finite)) if finite.size else np.nan


def finite_column_median(values: np.ndarray) -> np.ndarray:
    """Column medians without warnings for deliberately empty spectral bins."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 2:
        raise ValueError("finite_column_median requires a two-dimensional array.")
    result = np.full(values.shape[1], np.nan)
    represented = np.any(np.isfinite(values), axis=0)
    if np.any(represented):
        result[represented] = np.nanmedian(values[:, represented], axis=0)
    return result


def raw_height_csv(path: str | Path) -> np.ndarray:
    """Read one Keyence height grid without crop, fill, or leveling."""
    return (
        pd.read_csv(path, skiprows=19, header=None)
        .dropna(axis=1, how="all")
        .to_numpy(dtype=np.float64)
    )


def validate_and_fill_height(
    height: np.ndarray,
    *,
    expected_shape: tuple[int, int] | None = EXPECTED_EXP_SHAPE,
    max_missing_fraction: float = MAX_MISSING_FRACTION,
    max_missing_component_pixels: int = MAX_MISSING_COMPONENT_PIXELS,
) -> tuple[np.ndarray, dict[str, float | int]]:
    """Apply the acquisition-validity contract and fill isolated gaps."""
    height = np.asarray(height, dtype=np.float64)
    if expected_shape is not None and height.shape != expected_shape:
        raise ValueError(f"Expected height shape {expected_shape}, found {height.shape}.")

    missing = ~np.isfinite(height)
    missing_count = int(np.count_nonzero(missing))
    missing_fraction = missing_count / height.size
    if missing_fraction > max_missing_fraction:
        raise ValueError(
            f"Missing fraction {missing_fraction:.3g} exceeds {max_missing_fraction:.3g}."
        )

    largest_component = 0
    if missing_count:
        components, n_components = label(missing)
        if n_components:
            sizes = np.bincount(components.ravel())[1:]
            largest_component = int(sizes.max(initial=0))
        if largest_component > max_missing_component_pixels:
            raise ValueError(
                "Largest connected missing region is "
                f"{largest_component} pixels; limit is {max_missing_component_pixels}."
            )
        nearest = distance_transform_edt(
            missing,
            return_distances=False,
            return_indices=True,
        )
        height = height[tuple(nearest)]

    return height, {
        "missing_count": missing_count,
        "missing_fraction": float(missing_fraction),
        "largest_missing_component": largest_component,
    }


@lru_cache(maxsize=256)
def _detrend_geometry(
    shape: tuple[int, int],
    spacing_0_um: float,
    spacing_1_um: float,
    order: int,
) -> tuple[np.ndarray, np.ndarray]:
    row, column = np.indices(shape, dtype=np.float64)
    x0 = row.ravel() * float(spacing_0_um)
    x1 = column.ravel() * float(spacing_1_um)
    terms = [np.ones(row.size), x0, x1]
    if order == 2:
        terms.extend((x0 * x0, x0 * x1, x1 * x1))
    elif order != 1:
        raise ValueError("Detrend order must be 1 (plane) or 2 (quadratic).")
    design = np.column_stack(terms)
    return design, np.linalg.pinv(design)


def detrend_surface(
    values: np.ndarray,
    spacing_um: float | tuple[float, float],
    *,
    order: int = 1,
) -> np.ndarray:
    """Remove a least-squares plane (or named quadratic sensitivity)."""
    values = np.asarray(values, dtype=np.float64)
    if not np.all(np.isfinite(values)):
        raise ValueError("detrend_surface requires finite values.")
    if np.isscalar(spacing_um):
        d0 = d1 = float(spacing_um)
    else:
        d0, d1 = (float(v) for v in spacing_um)
    design, inverse = _detrend_geometry(values.shape, d0, d1, int(order))
    trend = (design @ (inverse @ values.ravel())).reshape(values.shape)
    return values - trend


def surface_metrics(height_um: np.ndarray) -> dict[str, float]:
    """Calculate standard scalar height metrics after mean removal."""
    z = np.asarray(height_um, dtype=np.float64)
    z = z - np.mean(z)
    sq = float(np.sqrt(np.mean(z**2)))
    return {
        "sa_um": float(np.mean(np.abs(z))),
        "sq_um": sq,
        "sz_robust_um": float(np.percentile(z, 99.5) - np.percentile(z, 0.5)),
        "ssk": float(np.mean(z**3) / sq**3) if sq > 0 else np.nan,
        "sku": float(np.mean(z**4) / sq**4) if sq > 0 else np.nan,
    }


def experimental_analysis_crop(
    raw_height_um: np.ndarray,
    *,
    crop: tuple[slice, slice] = EXP_ANALYSIS_CROP,
) -> np.ndarray:
    """Apply the sole experimental height-analysis crop."""
    raw_height_um = np.asarray(raw_height_um, dtype=float)
    if raw_height_um.shape != EXPECTED_EXP_SHAPE:
        raise ValueError(
            "Experimental analysis requires the validated raw map shape "
            f"{EXPECTED_EXP_SHAPE}; found {raw_height_um.shape}. This also prevents "
            "accidental application of the crop twice."
        )
    cropped = raw_height_um[crop]
    if cropped.ndim != 2 or min(cropped.shape) < 3:
        raise ValueError(f"Experimental analysis crop is empty/invalid: {cropped.shape}.")
    return cropped


def experimental_leveled_height(
    raw_height_um: np.ndarray,
    spacing_um: float,
    *,
    order: int = 1,
) -> np.ndarray:
    """Apply the experimental crop and level it for height analysis."""
    return detrend_surface(
        experimental_analysis_crop(raw_height_um),
        spacing_um,
        order=order,
    )


def resample_pixel_centers(
    values: np.ndarray,
    source_spacing_um: float | tuple[float, float],
    target_spacing_um: float | tuple[float, float] = TARGET_SPACING_UM,
) -> np.ndarray:
    """Bilinearly resample with centered pixel-cell geometry.

    Gaussian anti-aliasing is applied only along axes that are downsampled.
    Upsampling therefore does not invent additional measurement bandwidth.
    """
    values = np.asarray(values, dtype=np.float64)
    if np.isscalar(source_spacing_um):
        source = (float(source_spacing_um), float(source_spacing_um))
    else:
        source = tuple(float(v) for v in source_spacing_um)
    if np.isscalar(target_spacing_um):
        target = (float(target_spacing_um), float(target_spacing_um))
    else:
        target = tuple(float(v) for v in target_spacing_um)

    filtered = values
    sigma = []
    for source_d, target_d in zip(source, target):
        ratio = target_d / source_d
        sigma.append(0.5 * (ratio - 1.0) if ratio > 1.0 else 0.0)
    if any(value > 0 for value in sigma):
        filtered = gaussian_filter(filtered, sigma=sigma, mode="nearest")

    source_axes = [
        (np.arange(n, dtype=float) + 0.5) * spacing
        for n, spacing in zip(values.shape, source)
    ]
    target_axes = []
    for source_axis, target_d in zip(source_axes, target):
        # Interpolation is defined by the native pixel centers, not by values at
        # the outer pixel edges.  Center a target grid wholly inside that native
        # center support.  This avoids hidden edge extrapolation; after the
        # required analysis crop, a standard 10x map retains 21 matched windows.
        center_span = float(source_axis[-1] - source_axis[0])
        target_n = int(np.floor(center_span / target_d)) + 1
        used_center_span = (target_n - 1) * target_d
        offset = 0.5 * (center_span - used_center_span)
        target_axes.append(
            source_axis[0] + offset + np.arange(target_n, dtype=float) * target_d
        )

    interpolator = RegularGridInterpolator(
        tuple(source_axes),
        filtered,
        method="linear",
        bounds_error=True,
    )
    grid = np.meshgrid(*target_axes, indexing="ij")
    points = np.column_stack([axis.ravel() for axis in grid])
    return interpolator(points).reshape(tuple(len(axis) for axis in target_axes))


def _centered_nonoverlapping_windows(
    values: np.ndarray,
    window_shape_pixels: tuple[int, int],
) -> list[np.ndarray]:
    """Return the largest centered integer grid of non-overlapping windows."""
    values = np.asarray(values)
    w0, w1 = (int(v) for v in window_shape_pixels)
    n0 = values.shape[0] // w0
    n1 = values.shape[1] // w1
    if n0 < 1 or n1 < 1:
        raise ValueError(
            f"Array {values.shape} cannot contain a {window_shape_pixels} window."
        )
    used0, used1 = n0 * w0, n1 * w1
    start0 = (values.shape[0] - used0) // 2
    start1 = (values.shape[1] - used1) // 2
    return [
        values[
            start0 + i * w0 : start0 + (i + 1) * w0,
            start1 + j * w1 : start1 + (j + 1) * w1,
        ]
        for i in range(n0)
        for j in range(n1)
    ]


def manuscript_windows_256x128(
    raw_height_um: np.ndarray,
    source_spacing_um: float,
    *,
    source: str,
) -> list[np.ndarray]:
    """The sole sectioning operator: matched 256-by-128 um windows."""
    height = np.asarray(raw_height_um, dtype=np.float64)
    if not np.all(np.isfinite(height)):
        height, _ = validate_and_fill_height(height, expected_shape=None)

    if source == "exp":
        height = experimental_analysis_crop(height)
        height = resample_pixel_centers(
            height,
            source_spacing_um=source_spacing_um,
            target_spacing_um=TARGET_SPACING_UM,
        )
    elif source == "sim":
        if not np.isclose(source_spacing_um, TARGET_SPACING_UM):
            height = resample_pixel_centers(
                height,
                source_spacing_um=source_spacing_um,
                target_spacing_um=TARGET_SPACING_UM,
            )
    else:
        raise ValueError(f"Unknown source {source!r}.")

    shape_px = tuple(
        int(round(length / TARGET_SPACING_UM))
        for length in SPATIAL_WINDOW_SHAPE_UM
    )
    # SimResults should already return (z, transverse).  Permit the exact
    # transposed footprint, but do not silently reshape a genuinely different FOV.
    if source == "sim" and height.shape == shape_px[::-1]:
        height = height.T

    windows = _centered_nonoverlapping_windows(height, shape_px)
    prepared = []
    for window in windows:
        leveled = detrend_surface(window, TARGET_SPACING_UM, order=1)
        prepared.append(leveled - np.mean(leveled))
    return prepared




## Core PSD calculations


In [ ]:
def hann2d(shape: tuple[int, int]) -> np.ndarray:
    """Separable two-dimensional Hann taper."""
    return np.hanning(shape[0])[:, None] * np.hanning(shape[1])[None, :]


def periodogram_2d(
    height_um: np.ndarray,
    spacing_um: float | tuple[float, float],
) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    """Area-scaled 2D periodogram used by every PSD calculation.

    P(f0, f1) = d0*d1*|FFT(w*(h-mean(h)))|^2 / sum(w^2)
    """
    height = np.asarray(height_um, dtype=np.float64)
    if np.isscalar(spacing_um):
        d0 = d1 = float(spacing_um)
    else:
        d0, d1 = (float(v) for v in spacing_um)
    centered = height - np.mean(height)
    window = hann2d(height.shape)
    tapered = centered * window
    fft_height = np.fft.fft2(tapered)
    psd2d = d0 * d1 * np.abs(fft_height) ** 2 / np.sum(window**2)
    f0 = np.fft.fftfreq(height.shape[0], d=d0)
    f1 = np.fft.fftfreq(height.shape[1], d=d1)

    df0 = 1.0 / (height.shape[0] * d0)
    df1 = 1.0 / (height.shape[1] * d1)
    spectral_mean_square = float(np.sum(psd2d) * df0 * df1)
    real_mean_square = float(np.sum(tapered**2) / np.sum(window**2))
    denominator = max(real_mean_square, np.finfo(float).tiny)
    parseval_error = abs(spectral_mean_square - real_mean_square) / denominator
    return f0, f1, psd2d, float(parseval_error)


def matched_frequency_edges() -> np.ndarray:
    """17 fixed, equal-width physical-frequency annuli."""
    f_min = 1.0 / max(SPATIAL_WINDOW_SHAPE_UM)
    f_max = min(0.5 / TARGET_SPACING_UM, EXP_NATIVE_NYQUIST_UM_INV)
    return np.linspace(f_min, f_max, N_MATCHED_FREQUENCY_ANNULI + 1)


def legacy_frequency_area_normalize(
    frequency_um_inv: np.ndarray,
    radial_mean_um4: np.ndarray,
) -> np.ndarray:
    """Legacy manuscript-compatibility normalization.

    This makes the trapezoidal area of the annular-mean radial curve equal to
    one in frequency coordinates.  It does *not* equal unit resolved 2D power,
    because it omits annular mode multiplicity.  It is retained only to diagnose
    or reproduce the existing manuscript curve.
    """
    f = np.asarray(frequency_um_inv, dtype=float)
    y = np.asarray(radial_mean_um4, dtype=float)
    out = np.full_like(y, np.nan)
    valid = np.isfinite(f) & np.isfinite(y) & (f > 0) & (y > 0)
    if np.count_nonzero(valid) < 2:
        return out
    order = np.argsort(f[valid])
    area = trapezoid(y[valid][order], f[valid][order])
    if area > 0 and np.isfinite(area):
        out[valid] = y[valid] / area
    return out


def radial_psd(
    height_um: np.ndarray,
    spacing_um: float | tuple[float, float] = TARGET_SPACING_UM,
    *,
    frequency_edges_um_inv: np.ndarray | None = None,
    wavelength_min_um: float = WAVELENGTH_MIN_UM,
    wavelength_max_um: float = WAVELENGTH_MAX_UM,
) -> PSDResult:
    """Calculate raw radial PSD and two explicitly named normalizations.

    The canonical plotted PSD is a wavelength density: exact 2D mode power in
    each physical-frequency annulus is divided by total power in the configured
    wavelength band and by that bin's wavelength width.  Its integral on the
    requested *linear wavelength* axis is therefore one.  PSD gain remains a
    ratio of the unnormalized annular-mean ``radial_mean_um4`` values.
    """
    if not (0 < wavelength_min_um < wavelength_max_um):
        raise ValueError("Wavelength bounds must satisfy 0 < min < max.")
    if frequency_edges_um_inv is None:
        frequency_edges_um_inv = matched_frequency_edges()
    edges = np.asarray(frequency_edges_um_inv, dtype=float)
    if np.any(np.diff(edges) <= 0) or edges[0] <= 0:
        raise ValueError("Frequency edges must be finite, positive, and increasing.")

    f0, f1, psd2d, parseval_error = periodogram_2d(height_um, spacing_um)
    F1, F0 = np.meshgrid(f1, f0)
    radial_frequency = np.sqrt(F0**2 + F1**2)
    if np.isscalar(spacing_um):
        d0 = d1 = float(spacing_um)
    else:
        d0, d1 = (float(v) for v in spacing_um)
    df0 = 1.0 / (height_um.shape[0] * d0)
    df1 = 1.0 / (height_um.shape[1] * d1)

    band_f_min = 1.0 / wavelength_max_um
    band_f_max = 1.0 / wavelength_min_um
    n_bins = edges.size - 1
    frequency = np.full(n_bins, np.nan)
    wavelength = np.full(n_bins, np.nan)
    wavelength_lower = np.full(n_bins, np.nan)
    wavelength_upper = np.full(n_bins, np.nan)
    wavelength_width = np.full(n_bins, np.nan)
    radial_mean = np.full(n_bins, np.nan)
    annular_power = np.zeros(n_bins)
    modes = np.zeros(n_bins, dtype=int)

    for i, (edge_lo, edge_hi) in enumerate(zip(edges[:-1], edges[1:])):
        lo = max(float(edge_lo), band_f_min)
        hi = min(float(edge_hi), band_f_max)
        if hi <= lo:
            continue
        mask = (
            np.isfinite(radial_frequency)
            & np.isfinite(psd2d)
            & (radial_frequency >= lo)
            & (radial_frequency < hi)
        )
        modes[i] = int(np.count_nonzero(mask))
        if modes[i] == 0:
            continue
        # Report the mean modal frequency actually represented by the annulus;
        # bin edges still define the exact power and wavelength-density width.
        frequency[i] = float(np.mean(radial_frequency[mask]))
        lambda_lo = 1.0 / hi
        lambda_hi = 1.0 / lo
        wavelength[i] = 1.0 / frequency[i]
        wavelength_lower[i] = lambda_lo
        wavelength_upper[i] = lambda_hi
        wavelength_width[i] = lambda_hi - lambda_lo
        radial_mean[i] = float(np.mean(psd2d[mask]))
        annular_power[i] = float(np.sum(psd2d[mask]) * df0 * df1)

    total_power = float(np.sum(annular_power))
    wavelength_density = np.full(n_bins, np.nan)
    valid_power = (
        np.isfinite(wavelength_width)
        & (wavelength_width > 0)
        & (annular_power > 0)
        & (total_power > 0)
    )
    wavelength_density[valid_power] = (
        annular_power[valid_power] / total_power / wavelength_width[valid_power]
    )
    legacy = legacy_frequency_area_normalize(frequency, radial_mean)

    # Descriptors follow the protocol's complete matched-mode support rather
    # than the narrower configurable display interval.  This distinction is
    # necessary for spectral medians longer than WAVELENGTH_MAX_UM (the
    # manuscript reports values near 71 um).
    descriptor_f_min = float(edges[0])
    descriptor_f_max = float(edges[-1])
    resolved = (
        np.isfinite(radial_frequency)
        & np.isfinite(psd2d)
        & (radial_frequency >= descriptor_f_min)
        & (radial_frequency <= descriptor_f_max)
        & (radial_frequency > 0)
        & (psd2d >= 0)
    )
    resolved_frequency = radial_frequency[resolved]
    resolved_power = psd2d[resolved] * df0 * df1
    spectral_median = np.nan
    long_fraction = np.nan
    if resolved_power.size and np.sum(resolved_power) > 0:
        resolved_wavelength = 1.0 / resolved_frequency
        order = np.argsort(resolved_wavelength)
        cumulative = np.cumsum(resolved_power[order])
        median_index = int(np.searchsorted(cumulative, 0.5 * cumulative[-1]))
        spectral_median = float(resolved_wavelength[order][median_index])
        long_fraction = float(
            np.sum(resolved_power[resolved_wavelength >= 32.0])
            / np.sum(resolved_power)
        )

    descriptor_frequency = np.full(n_bins, np.nan)
    descriptor_radial_mean = np.full(n_bins, np.nan)
    for index, (lower, upper) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (
            np.isfinite(radial_frequency)
            & np.isfinite(psd2d)
            & (radial_frequency >= lower)
            & (radial_frequency < upper)
        )
        if np.any(mask):
            descriptor_frequency[index] = float(np.mean(radial_frequency[mask]))
            descriptor_radial_mean[index] = float(np.mean(psd2d[mask]))
    exponent_mask = (
        np.isfinite(descriptor_frequency)
        & np.isfinite(descriptor_radial_mean)
        & (descriptor_frequency > 0)
        & (descriptor_radial_mean > 0)
        & (1.0 / descriptor_frequency <= 32.0)
    )
    high_frequency_exponent = np.nan
    if np.count_nonzero(exponent_mask) >= 3:
        high_frequency_exponent = float(
            np.polyfit(
                np.log10(descriptor_frequency[exponent_mask]),
                np.log10(descriptor_radial_mean[exponent_mask]),
                1,
            )[0]
        )
    return PSDResult(
        frequency_um_inv=frequency,
        wavelength_um=wavelength,
        wavelength_lower_um=wavelength_lower,
        wavelength_upper_um=wavelength_upper,
        wavelength_width_um=wavelength_width,
        radial_mean_um4=radial_mean,
        annular_power_um2=annular_power,
        normalized_wavelength_density_um_inv=wavelength_density,
        legacy_frequency_area_normalized_um=legacy,
        modes=modes,
        parseval_relative_error=parseval_error,
        spectral_median_wavelength_um=spectral_median,
        long_wavelength_power_fraction=long_fraction,
        high_frequency_exponent=high_frequency_exponent,
    )


def psd_gain_decades(initial: PSDResult, final: PSDResult) -> np.ndarray:
    """Paired absolute PSD gain, log10(C_final/C_initial)."""
    if not np.allclose(
        initial.frequency_um_inv,
        final.frequency_um_inv,
        equal_nan=True,
    ):
        raise ValueError("Initial and final frequency grids differ.")
    out = np.full_like(initial.radial_mean_um4, np.nan)
    valid = (
        np.isfinite(initial.radial_mean_um4)
        & np.isfinite(final.radial_mean_um4)
        & (initial.radial_mean_um4 > 0)
        & (final.radial_mean_um4 > 0)
    )
    out[valid] = np.log10(
        final.radial_mean_um4[valid] / initial.radial_mean_um4[valid]
    )
    return out




## Core ACF calculations


In [ ]:
def overlap_corrected_acf_2d(height_um: np.ndarray) -> np.ndarray:
    """Linear overlap-corrected covariance normalized at zero lag."""
    height = np.asarray(height_um, dtype=np.float64)
    finite = np.isfinite(height)
    if np.count_nonzero(finite) < 10:
        raise ValueError("Too few finite values for ACF.")
    centered = np.zeros_like(height)
    centered[finite] = height[finite] - np.mean(height[finite])
    finite_float = finite.astype(float)
    numerator = fftconvolve(centered, centered[::-1, ::-1], mode="full")
    overlap = fftconvolve(finite_float, finite_float[::-1, ::-1], mode="full")
    # True overlap counts are integers.  FFT roundoff otherwise turns exact
        # zero-overlap lags into tiny positive values and can create extreme ratios.
    overlap = np.rint(overlap)
    covariance = np.full_like(numerator, np.nan)
    valid = overlap >= 1.0
    covariance[valid] = numerator[valid] / overlap[valid]
    center = tuple(np.asarray(covariance.shape) // 2)
    zero_lag = covariance[center]
    if not np.isfinite(zero_lag) or zero_lag <= 0:
        raise ValueError("ACF zero-lag covariance is not positive.")
    return covariance / zero_lag


def first_crossing(
    x: np.ndarray,
    y: np.ndarray,
    target: float,
) -> float:
    """First downward threshold crossing with linear interpolation."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    order = np.argsort(x)
    x, y = x[order], y[order]
    for i in range(1, len(x)):
        if y[i - 1] > target and y[i] <= target:
            if y[i] == y[i - 1]:
                return float(x[i])
            fraction = (target - y[i - 1]) / (y[i] - y[i - 1])
            return float(x[i - 1] + fraction * (x[i] - x[i - 1]))
    return np.nan


def radial_acf(
    height_um: np.ndarray,
    spacing_um: float | tuple[float, float] = TARGET_SPACING_UM,
    *,
    bin_width_um: float = ACF_RADIAL_BIN_WIDTH_UM,
    max_lag_um: float = ACF_MAX_LAG_UM,
) -> ACFResult:
    """Radially average the canonical linear ACF in physical annuli."""
    if np.isscalar(spacing_um):
        d0 = d1 = float(spacing_um)
    else:
        d0, d1 = (float(v) for v in spacing_um)
    rho = overlap_corrected_acf_2d(height_um)
    lag0 = np.arange(-(height_um.shape[0] - 1), height_um.shape[0]) * d0
    lag1 = np.arange(-(height_um.shape[1] - 1), height_um.shape[1]) * d1
    L1, L0 = np.meshgrid(lag1, lag0)
    radius = np.sqrt(L0**2 + L1**2)

    centers = np.arange(0.0, max_lag_um + 0.5 * bin_width_um, bin_width_um)
    edges = np.concatenate(
        ([0.0], centers[1:] - 0.5 * bin_width_um, [centers[-1] + 0.5 * bin_width_um])
    )
    radial = np.full(centers.size, np.nan)
    counts = np.zeros(centers.size, dtype=int)
    for i in range(centers.size):
        mask = (
            np.isfinite(radius)
            & np.isfinite(rho)
            & (radius >= edges[i])
            & (radius < edges[i + 1])
        )
        counts[i] = int(np.count_nonzero(mask))
        if counts[i]:
            radial[i] = float(np.mean(rho[mask]))
    radial[0] = 1.0
    one_over_e = first_crossing(centers, radial, np.exp(-1.0))
    first_zero = first_crossing(centers, radial, 0.0)
    return ACFResult(
        lag_um=centers,
        correlation=radial,
        counts=counts,
        one_over_e_um=one_over_e,
        first_zero_um=first_zero,
        one_over_e_crossed=bool(np.isfinite(one_over_e)),
        first_zero_crossed=bool(np.isfinite(first_zero)),
    )


def map_spatial_curves(
    raw_height_um: np.ndarray,
    spacing_um: float,
    *,
    source: str,
) -> dict[str, np.ndarray | float | int]:
    """Reduce matched window descriptors to one map/specimen observation."""
    windows = manuscript_windows_256x128(raw_height_um, spacing_um, source=source)
    psd_results = [radial_psd(window) for window in windows]
    acf_results = [radial_acf(window) for window in windows]

    raw_psd = np.vstack([result.radial_mean_um4 for result in psd_results])
    power = np.vstack([result.annular_power_um2 for result in psd_results])
    acf = np.vstack([result.correlation for result in acf_results])
    one_over_e = np.asarray([result.one_over_e_um for result in acf_results])
    first_zero = np.asarray([result.first_zero_um for result in acf_results])
    spectral_median = np.asarray(
        [result.spectral_median_wavelength_um for result in psd_results]
    )
    long_fraction = np.asarray(
        [result.long_wavelength_power_fraction for result in psd_results]
    )
    high_frequency_exponent = np.asarray(
        [result.high_frequency_exponent for result in psd_results]
    )
    reference_psd = psd_results[0]
    reference_acf = acf_results[0]
    map_raw_psd = finite_column_median(raw_psd)
    map_power = np.nanmedian(power, axis=0)
    total_power = np.nansum(map_power)
    map_density = np.full_like(map_power, np.nan)
    valid_density = (
        np.isfinite(reference_psd.wavelength_width_um)
        & (reference_psd.wavelength_width_um > 0)
        & (map_power > 0)
        & (total_power > 0)
    )
    map_density[valid_density] = (
        map_power[valid_density]
        / total_power
        / reference_psd.wavelength_width_um[valid_density]
    )
    return {
        "frequency_um_inv": reference_psd.frequency_um_inv,
        "wavelength_um": reference_psd.wavelength_um,
        "wavelength_lower_um": reference_psd.wavelength_lower_um,
        "wavelength_upper_um": reference_psd.wavelength_upper_um,
        "wavelength_width_um": reference_psd.wavelength_width_um,
        "radial_psd_um4": map_raw_psd,
        "annular_power_um2": map_power,
        "normalized_psd_um_inv": map_density,
        "legacy_normalized_psd_um": legacy_frequency_area_normalize(
            reference_psd.frequency_um_inv,
            map_raw_psd,
        ),
        "acf_lag_um": reference_acf.lag_um,
        "acf": finite_column_median(acf),
        # Scalar landmark is median-of-window crossings, not crossing-of-median.
        "acf_one_over_e_um": finite_median(one_over_e),
        "acf_first_zero_um": finite_median(first_zero),
        "acf_one_over_e_crossed_fraction": float(np.mean(np.isfinite(one_over_e))),
        "acf_first_zero_crossed_fraction": float(np.mean(np.isfinite(first_zero))),
        "spectral_median_wavelength_um": finite_median(spectral_median),
        "long_wavelength_power_fraction": finite_median(long_fraction),
        "high_frequency_exponent": finite_median(high_frequency_exponent),
        "n_windows": len(windows),
    }


def nonsectioned_spatial_curves(
    height_um: np.ndarray,
    spacing_um: float,
    *,
    acf_max_lag_um: float = ACF_MAX_LAG_UM,
) -> dict[str, np.ndarray | float]:
    """PSD and ACF of one whole supplied field, with no internal sectioning.

    Experimental callers must supply the fixed analysis crop; simulation callers
    supply the complete unique boundary face.  This function intentionally does
    not call the sectioning operator.
    """
    selected = np.asarray(height_um, dtype=float)
    leveled = detrend_surface(selected, spacing_um, order=1)
    leveled -= np.mean(leveled)
    psd = radial_psd(
        leveled,
        spacing_um,
        wavelength_min_um=WAVELENGTH_MIN_UM,
        wavelength_max_um=WAVELENGTH_MAX_UM,
    )
    acf = radial_acf(
        leveled,
        spacing_um,
        bin_width_um=spacing_um,
        max_lag_um=acf_max_lag_um,
    )
    return {
        "height_um": leveled,
        "frequency_um_inv": psd.frequency_um_inv,
        "wavelength_um": psd.wavelength_um,
        "wavelength_lower_um": psd.wavelength_lower_um,
        "wavelength_upper_um": psd.wavelength_upper_um,
        "radial_psd_um4": psd.radial_mean_um4,
        "normalized_psd_um_inv": psd.normalized_wavelength_density_um_inv,
        "legacy_normalized_psd_um": psd.legacy_frequency_area_normalized_um,
        "annular_power_um2": psd.annular_power_um2,
        "wavelength_width_um": psd.wavelength_width_um,
        "parseval_relative_error": psd.parseval_relative_error,
        "acf_lag_um": acf.lag_um,
        "acf": acf.correlation,
        "acf_one_over_e_um": acf.one_over_e_um,
        "acf_first_zero_um": acf.first_zero_um,
        "acf_one_over_e_crossed": acf.one_over_e_crossed,
        "acf_first_zero_crossed": acf.first_zero_crossed,
        "spectral_median_wavelength_um": psd.spectral_median_wavelength_um,
        "long_wavelength_power_fraction": psd.long_wavelength_power_fraction,
        "high_frequency_exponent": psd.high_frequency_exponent,
    }


def directional_line_roughness(height_um: np.ndarray) -> dict[str, float]:
    """Mean line roughness parallel and transverse to the loading direction."""
    height = np.asarray(height_um, dtype=float)
    along_z = height - np.mean(height, axis=0, keepdims=True)
    along_y = height - np.mean(height, axis=1, keepdims=True)
    ra_z = float(np.mean(np.mean(np.abs(along_z), axis=0)))
    ra_y = float(np.mean(np.mean(np.abs(along_y), axis=1)))
    rq_z = float(np.mean(np.sqrt(np.mean(along_z**2, axis=0))))
    rq_y = float(np.mean(np.sqrt(np.mean(along_y**2, axis=1))))
    return {
        "ra_parallel_z_um": ra_z,
        "ra_transverse_y_um": ra_y,
        "rq_parallel_z_um": rq_z,
        "rq_transverse_y_um": rq_y,
        "ra_anisotropy": ra_z / ra_y if ra_y > 0 else np.nan,
        "rq_anisotropy": rq_z / rq_y if rq_y > 0 else np.nan,
    }




## One-time raw-data caches


In [ ]:
def _numeric_file_sort_key(path: Path) -> tuple[int, float, str]:
    try:
        return 0, float(path.stem), str(path)
    except ValueError:
        return 1, np.inf, str(path)


def _array_key(*parts: object) -> str:
    text = "|".join(str(part) for part in parts)
    return "h_" + hashlib.sha1(text.encode("utf-8")).hexdigest()[:20]


class StreamingNpzWriter(AbstractContextManager["StreamingNpzWriter"]):
    """Write a large NPZ incrementally instead of retaining every map in RAM."""

    def __init__(self, destination: Path):
        self.destination = Path(destination)
        self.temporary = self.destination.with_suffix(self.destination.suffix + ".tmp")
        self.archive: zipfile.ZipFile | None = None

    def __enter__(self) -> "StreamingNpzWriter":
        self.destination.parent.mkdir(parents=True, exist_ok=True)
        self.temporary.unlink(missing_ok=True)
        self.archive = zipfile.ZipFile(
            self.temporary,
            mode="w",
            compression=zipfile.ZIP_DEFLATED,
            allowZip64=True,
        )
        return self

    def write(self, key: str, values: np.ndarray) -> None:
        if self.archive is None:
            raise RuntimeError("StreamingNpzWriter is not open.")
        with self.archive.open(f"{key}.npy", mode="w", force_zip64=True) as stream:
            np.lib.format.write_array(
                stream,
                np.ascontiguousarray(values),
                allow_pickle=False,
            )

    def __exit__(self, exc_type, exc, traceback) -> bool:
        if self.archive is not None:
            self.archive.close()
        if exc_type is None:
            self.temporary.replace(self.destination)
        else:
            self.temporary.unlink(missing_ok=True)
        return False


def _write_dataframe_cache(frame: pd.DataFrame, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    frame.to_pickle(temporary)
    temporary.replace(destination)


def _cache_configuration() -> dict[str, object]:
    """Return the raw-cache contract in JSON-serializable form."""
    return {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "data_dir": str(Path(DATA_DIR).resolve()),
        "microstructure_dir": str(Path(MICROSTRUCTURE_DIR).resolve()),
        "experiments": [[int(load), sample_type] for load, sample_type in EXPERIMENTS],
        "magnification": MAGNIFICATION,
        "polish": POLISH,
        "experimental_native_spacing_um": EXP_NATIVE_SPACING_UM,
        "simulation_spacing_um": float(VOXELSIZE),
        "expected_experimental_shape": list(EXPECTED_EXP_SHAPE),
        "simulation_runs": [
            {
                "micro_id": str(run["micro_id"]),
                "sim_root": str(Path(run["sim_root"])),
                "microstructure": str(Path(run["microstructure"])),
            }
            for run in MICRO_RUNS
        ],
    }


def _write_cache_manifest() -> None:
    temporary = CACHE_MANIFEST.with_suffix(CACHE_MANIFEST.suffix + ".tmp")
    temporary.write_text(
        json.dumps(_cache_configuration(), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(CACHE_MANIFEST)


def _cache_manifest_is_current() -> bool:
    try:
        recorded = json.loads(CACHE_MANIFEST.read_text(encoding="utf-8"))
    except (OSError, ValueError, TypeError):
        return False
    return recorded == _cache_configuration()


def _read_strain_table(load_mpa: int, sample_type: str) -> pd.DataFrame:
    path = Path(DATA_DIR) / f"creep_{sample_type}_{POLISH}_{load_mpa}" / "strain.csv"
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    frame.columns = [str(column).strip() for column in frame.columns]
    if "time_h" not in frame:
        warnings.warn(f"No time_h column in {path}.")
        return pd.DataFrame()
    for column in frame:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def _bounded_interpolation(
    x: np.ndarray,
    y: np.ndarray,
    query: float,
) -> float:
    """Interpolate only inside measured support; never clamp/extrapolate."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if np.count_nonzero(valid) < 1:
        return np.nan
    x, y = x[valid], y[valid]
    order = np.argsort(x)
    x, y = x[order], y[order]
    if query < x[0] or query > x[-1]:
        return np.nan
    return float(np.interp(query, x, y))


def _experimental_strain_history(
    strain_df: pd.DataFrame,
    sample_id: str,
) -> tuple[np.ndarray, np.ndarray, str]:
    if strain_df.empty:
        return np.empty(0), np.empty(0), "none"
    time = strain_df["time_h"].to_numpy(dtype=float)
    candidates = [column for column in strain_df.columns if column != "time_h"]
    lookup = {str(column).strip().lower(): column for column in candidates}
    match = lookup.get(str(sample_id).strip().lower())
    if match is not None:
        strain = strain_df[match].to_numpy(dtype=float)
        source = "sample"
    elif candidates:
        strain = np.nanmean(strain_df[candidates].to_numpy(dtype=float), axis=1)
        source = "experiment_mean"
    else:
        return np.empty(0), np.empty(0), "none"
    valid = np.isfinite(time) & np.isfinite(strain)
    if not np.any(valid):
        return np.empty(0), np.empty(0), "none"
    order = np.argsort(time[valid])
    return time[valid][order], strain[valid][order], source


def _experimental_strain_at_time(
    strain_df: pd.DataFrame,
    sample_id: str,
    time_h: float,
) -> tuple[float, str]:
    time, strain, source = _experimental_strain_history(strain_df, sample_id)
    if time.size:
        value = _bounded_interpolation(time, strain, time_h)
        if np.isfinite(value):
            return value, source
    return np.nan, "none"


def build_experimental_cache() -> tuple[pd.DataFrame, Mapping[str, np.ndarray]]:
    """Read every acquisition-valid experimental map exactly once."""
    rows: list[dict[str, object]] = []
    with StreamingNpzWriter(EXP_HEIGHTS_CACHE) as writer:
        for load_mpa, sample_type in EXPERIMENTS:
            root = (
                Path(DATA_DIR)
                / f"creep_{sample_type}_{POLISH}_{load_mpa}"
                / "profilometry"
                / MAGNIFICATION
            )
            if not root.exists():
                warnings.warn(f"Missing profilometry directory: {root}")
                continue
            strain_df = _read_strain_table(load_mpa, sample_type)
            for path in sorted(root.rglob("*.csv"), key=_numeric_file_sort_key):
                relative = path.relative_to(root)
                if len(relative.parts) < 2:
                    continue
                if any(part in EXCLUDED_DIRECTORY_NAMES for part in relative.parts):
                    continue
                try:
                    time_h = float(path.stem)
                except ValueError:
                    continue
                sample_id = str(relative.parts[0]).strip()
                try:
                    raw, acquisition = validate_and_fill_height(raw_height_csv(path))
                except Exception as exc:
                    warnings.warn(f"Rejected {path}: {exc}")
                    continue
                key = _array_key("exp", load_mpa, sample_type, sample_id, time_h, path)
                writer.write(key, raw.astype(np.float32))
                strain, strain_source = _experimental_strain_at_time(
                    strain_df,
                    sample_id,
                    time_h,
                )
                mechanical_time_h, mechanical_strain, mechanical_source = (
                    _experimental_strain_history(strain_df, sample_id)
                )
                rows.append(
                    {
                        "cache_schema_version": CACHE_SCHEMA_VERSION,
                        "source": "exp",
                        "load_mpa": int(load_mpa),
                        "sample_type": sample_type,
                        "polish": POLISH,
                        "magnification": MAGNIFICATION,
                        "sample_id": sample_id,
                        "replicate_id": sample_id,
                        "time_h": time_h,
                        "bulk_z_strain": strain,
                        "bulk_z_strain_percent": 100.0 * strain,
                        "strain_source": strain_source,
                        "mechanical_time_h": mechanical_time_h,
                        "mechanical_bulk_z_strain_percent": 100.0 * mechanical_strain,
                        "mechanical_history_source": mechanical_source,
                        "height_key": key,
                        "height_path": str(path),
                        "spacing_um": EXP_NATIVE_SPACING_UM,
                        "shape_0": raw.shape[0],
                        "shape_1": raw.shape[1],
                        # 588 remains available for descriptive endpoint plots,
                        # but not primary paired/cross-load inference.
                        "primary_paired_spatial_eligible": load_mpa != 588,
                        **acquisition,
                    }
                )

    exp_df = pd.DataFrame(rows)
    if exp_df.empty:
        raise RuntimeError("No experimental maps were cached.")
    groups = exp_df.groupby(["load_mpa", "sample_type", "sample_id"])["time_h"]
    exp_df["is_initial"] = exp_df["time_h"].eq(groups.transform("min"))
    exp_df["is_endpoint"] = exp_df["time_h"].eq(groups.transform("max"))
    group_count = groups.transform("size")
    group_min_time = groups.transform("min")
    exp_df["has_paired_baseline_endpoint"] = (
        (group_count >= 2) & np.isclose(group_min_time, 0.0, atol=1.0e-8)
    )
    exp_df["primary_paired_spatial_eligible"] &= exp_df[
        "has_paired_baseline_endpoint"
    ]
    exp_df = exp_df.sort_values(
        ["load_mpa", "sample_type", "sample_id", "time_h"]
    ).reset_index(drop=True)
    _write_dataframe_cache(exp_df, EXP_DF_CACHE)
    _write_cache_manifest()
    return exp_df, np.load(EXP_HEIGHTS_CACHE, allow_pickle=False)


def build_simulation_cache() -> tuple[pd.DataFrame, Mapping[str, np.ndarray]]:
    """Read every required SimResults run once and cache all height states."""
    rows: list[dict[str, object]] = []
    with StreamingNpzWriter(SIM_HEIGHTS_CACHE) as writer:
        for run in MICRO_RUNS:
            micro_id = str(run["micro_id"])
            for load_mpa, sample_type in EXPERIMENTS:
                run_dir = Path(run["sim_root"]) / f"{load_mpa}mpa_{sample_type}"
                try:
                    result = SimResults.load(
                        run_dir,
                        microstructure=Path(run["microstructure"]),
                    )
                except Exception as exc:
                    warnings.warn(f"Could not load {run_dir}: {exc}")
                    continue
                height = np.asarray(result.height, dtype=float)
                if height.ndim != 4:
                    warnings.warn(
                        f"Expected (face,time,z,width) height data in {run_dir}; "
                        f"found {height.shape}."
                    )
                    continue
                vtk_time = np.asarray(result.vtk_time, dtype=float)
                sim_time = np.asarray(result.sim_time, dtype=float)
                bulk_strain = np.asarray(result.epav33, dtype=float)
                valid = np.isfinite(sim_time) & np.isfinite(bulk_strain)
                sim_time_valid = sim_time[valid]
                strain_valid = bulk_strain[valid]
                order = np.argsort(sim_time_valid)
                sim_time_valid = sim_time_valid[order]
                strain_valid = strain_valid[order]
                n_times = min(height.shape[1], vtk_time.size)
                face_names = getattr(result, "samples", None)
                for face_index in range(height.shape[0]):
                    face_name = (
                        str(face_names[face_index])
                        if face_names is not None and face_index < len(face_names)
                        else f"face_{face_index}"
                    )
                    for time_index in range(n_times):
                        time_s = float(vtk_time[time_index])
                        strain = _bounded_interpolation(
                            sim_time_valid,
                            strain_valid,
                            time_s,
                        )
                        array = np.asarray(height[face_index, time_index], dtype=float)
                        if not np.all(np.isfinite(array)):
                            array, _ = validate_and_fill_height(
                                array,
                                expected_shape=None,
                                max_missing_fraction=1.0,
                                max_missing_component_pixels=array.size,
                            )
                        key = _array_key(
                            "sim",
                            micro_id,
                            load_mpa,
                            sample_type,
                            face_index,
                            time_index,
                        )
                        writer.write(key, array.astype(np.float32))
                        rows.append(
                            {
                                "cache_schema_version": CACHE_SCHEMA_VERSION,
                                "source": "sim",
                                "micro_id": micro_id,
                                "load_mpa": int(load_mpa),
                                "sample_type": sample_type,
                                "sample_id": micro_id,
                                "replicate_id": micro_id,
                                "face_index": face_index,
                                "face_name": face_name,
                                "unique_plane": face_index in SIM_UNIQUE_FACE_INDICES,
                                "time_index": time_index,
                                "time_s": time_s,
                                "time_h": time_s / 3600.0,
                                "bulk_z_strain": strain,
                                "bulk_z_strain_percent": 100.0 * strain,
                                "mechanical_time_h": sim_time_valid / 3600.0,
                                "mechanical_bulk_z_strain_percent": 100.0
                                * strain_valid,
                                "height_key": key,
                                "spacing_um": float(VOXELSIZE),
                                "shape_0": array.shape[0],
                                "shape_1": array.shape[1],
                                "run_dir": str(run_dir),
                                "microstructure": str(run["microstructure"]),
                            }
                        )

    sim_df = pd.DataFrame(rows)
    if sim_df.empty:
        raise RuntimeError("No simulation maps were cached.")
    groups = sim_df.groupby(
        ["micro_id", "load_mpa", "sample_type", "face_index"]
    )["time_index"]
    sim_df["is_initial"] = sim_df["time_index"].eq(groups.transform("min"))
    sim_df["is_endpoint"] = sim_df["time_index"].eq(groups.transform("max"))
    sim_df = sim_df.sort_values(
        ["micro_id", "load_mpa", "sample_type", "face_index", "time_index"]
    ).reset_index(drop=True)
    _write_dataframe_cache(sim_df, SIM_DF_CACHE)
    _write_cache_manifest()
    return sim_df, np.load(SIM_HEIGHTS_CACHE, allow_pickle=False)


def _cache_is_current(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        frame = pd.read_pickle(path)
    except Exception:
        return False
    return (
        _cache_manifest_is_current()
        and
        "cache_schema_version" in frame
        and len(frame) > 0
        and frame["cache_schema_version"].eq(CACHE_SCHEMA_VERSION).all()
    )


def load_experimental_cache(
    *,
    rebuild: bool = False,
) -> tuple[pd.DataFrame, Mapping[str, np.ndarray]]:
    if rebuild or not (
        _cache_is_current(EXP_DF_CACHE) and EXP_HEIGHTS_CACHE.exists()
    ):
        return build_experimental_cache()
    return (
        pd.read_pickle(EXP_DF_CACHE),
        np.load(EXP_HEIGHTS_CACHE, allow_pickle=False),
    )


def load_simulation_cache(
    *,
    rebuild: bool = False,
) -> tuple[pd.DataFrame, Mapping[str, np.ndarray]]:
    if rebuild or not (
        _cache_is_current(SIM_DF_CACHE) and SIM_HEIGHTS_CACHE.exists()
    ):
        return build_simulation_cache()
    return (
        pd.read_pickle(SIM_DF_CACHE),
        np.load(SIM_HEIGHTS_CACHE, allow_pickle=False),
    )


def load_all_caches(
    *,
    rebuild: bool = False,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    Mapping[str, np.ndarray],
    Mapping[str, np.ndarray],
]:
    """Return the only four raw-data objects used in later cells."""
    exp_df, exp_heights = load_experimental_cache(rebuild=rebuild)
    sim_df, sim_heights = load_simulation_cache(rebuild=rebuild)
    return exp_df, sim_df, exp_heights, sim_heights


# Exact notebook-facing cache names.  ``initialize_caches`` populates these once
# per Python process; every later cell consumes the same four objects.
exp_df: pd.DataFrame | None = None
sim_df: pd.DataFrame | None = None
exp_heights: Mapping[str, np.ndarray] | None = None
sim_heights: Mapping[str, np.ndarray] | None = None


def initialize_caches(
    *,
    rebuild: bool = False,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    Mapping[str, np.ndarray],
    Mapping[str, np.ndarray],
]:
    """Load/build the four caches at most once unless rebuilding is explicit."""
    global exp_df, sim_df, exp_heights, sim_heights
    if rebuild or any(
        value is None for value in (exp_df, sim_df, exp_heights, sim_heights)
    ):
        for heights in (exp_heights, sim_heights):
            close = getattr(heights, "close", None)
            if callable(close):
                close()
        exp_df, sim_df, exp_heights, sim_heights = load_all_caches(
            rebuild=rebuild
        )
    return exp_df, sim_df, exp_heights, sim_heights




## Derived scalar tables (pure transformations of the four caches)


In [ ]:
def experimental_roughness_table(
    exp_df: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
) -> pd.DataFrame:
    """Calculate every experimental height metric on the fixed analysis crop."""
    rows = []
    for record in exp_df.itertuples(index=False):
        height = np.asarray(exp_heights[record.height_key], dtype=float)
        cropped_leveled = experimental_leveled_height(
            height,
            float(record.spacing_um),
        )
        metrics = surface_metrics(cropped_leveled)
        directionality = directional_line_roughness(cropped_leveled)
        rows.append(
            {
                **record._asdict(),
                **metrics,
                **directionality,
                "analysis_domain": "analysis_crop",
            }
        )
    table = pd.DataFrame(rows)
    group = table.groupby(["load_mpa", "sample_type", "sample_id"])
    table["sa_initial_um"] = group["sa_um"].transform("first")
    table["delta_sa_um"] = table["sa_um"] - table["sa_initial_um"]
    for column in ("sq_um", "sz_robust_um"):
        table[f"{column.removesuffix('_um')}_initial_um"] = group[column].transform(
            "first"
        )
        table[f"delta_{column}"] = (
            table[column] - table[f"{column.removesuffix('_um')}_initial_um"]
        )
    return table


def simulation_roughness_table(
    sim_df: pd.DataFrame,
    sim_heights: Mapping[str, np.ndarray],
) -> pd.DataFrame:
    """Add plane-levelled boundary Sa without treating faces as replicates."""
    rows = []
    for record in sim_df.itertuples(index=False):
        height = np.asarray(sim_heights[record.height_key], dtype=float)
        leveled = detrend_surface(height, float(record.spacing_um), order=1)
        rows.append({**record._asdict(), **surface_metrics(leveled)})
    table = pd.DataFrame(rows)
    group = table.groupby(
        ["micro_id", "load_mpa", "sample_type", "face_index"]
    )
    table["sa_initial_um"] = group["sa_um"].transform("first")
    table["delta_sa_um"] = table["sa_um"] - table["sa_initial_um"]
    return table


def aggregate_simulation_hierarchy(
    table: pd.DataFrame,
    *,
    group_columns: Sequence[str],
    value_columns: Sequence[str],
) -> pd.DataFrame:
    """Mean unique faces within realization, then realizations within case."""
    selected = table[
        table["unique_plane"]
        & table["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
    ].copy()
    face_mean = (
        selected.groupby([*group_columns, "micro_id"], as_index=False)[
            list(value_columns)
        ]
        .mean()
    )
    return face_mean.groupby(list(group_columns), as_index=False)[
        list(value_columns)
    ].mean()


def assign_strain_groups(
    frame: pd.DataFrame,
    *,
    n_groups: int = 4,
    strain_column: str = "bulk_z_strain_percent",
    include_initial: bool = True,
) -> tuple[pd.DataFrame, list[str], dict[str, np.ndarray]]:
    """Assign ordered quantile groups and their sole viridis color mapping."""
    result = frame.copy()
    result["strain_group"] = pd.NA
    positive = result[np.isfinite(result[strain_column]) & (result[strain_column] > 0)]
    if positive.empty:
        return result.iloc[0:0], [], {}
    quantiles = np.unique(
        np.quantile(positive[strain_column], np.linspace(0, 1, n_groups + 1))
    )
    labels: list[str] = []
    if include_initial:
        result.loc[result[strain_column] <= 0, "strain_group"] = "initial"
        labels.append("initial")
    positive_labels = []
    for index, (low, high) in enumerate(zip(quantiles[:-1], quantiles[1:])):
        label_text = f"{low:.2f}-{high:.2f}%"
        positive_labels.append(label_text)
        if index == len(quantiles) - 2:
            mask = (result[strain_column] >= low) & (result[strain_column] <= high)
        else:
            mask = (result[strain_column] >= low) & (result[strain_column] < high)
        result.loc[mask, "strain_group"] = label_text
    labels.extend(positive_labels)
    colors = strain_group_colors(len(labels))
    color_map: dict[str, np.ndarray] = {
        label_text: color for label_text, color in zip(labels, colors)
    }
    return result[result["strain_group"].notna()].copy(), labels, color_map


def mechanical_history_table(
    exp_df: pd.DataFrame,
    sim_df: pd.DataFrame,
) -> pd.DataFrame:
    """Unpack cached mechanical histories without reopening raw result files."""
    rows: list[dict[str, object]] = []
    for source, frame, keys in (
        ("exp", exp_df, ["load_mpa", "sample_type", "sample_id"]),
        ("sim", sim_df, ["load_mpa", "sample_type", "micro_id"]),
    ):
        required = {
            *keys,
            "mechanical_time_h",
            "mechanical_bulk_z_strain_percent",
        }
        if not required.issubset(frame.columns):
            continue
        for identity, group in frame.groupby(keys, dropna=False):
            record = group.iloc[0]
            time_h = np.asarray(record["mechanical_time_h"], dtype=float)
            strain = np.asarray(
                record["mechanical_bulk_z_strain_percent"], dtype=float
            )
            if time_h.size != strain.size:
                raise ValueError(f"Cached {source} mechanical history sizes differ.")
            identity_values = identity if isinstance(identity, tuple) else (identity,)
            metadata = dict(zip(keys, identity_values))
            for time_value, strain_value in zip(time_h, strain):
                if np.isfinite(time_value) and np.isfinite(strain_value):
                    rows.append(
                        {
                            "source": source,
                            **metadata,
                            "time_h": float(time_value),
                            "bulk_z_strain_percent": float(strain_value),
                        }
                    )
    return pd.DataFrame(rows)


def validated_wavelength_bands(
    bands_um: Mapping[str, tuple[float, float]] = DEFAULT_WAVELENGTH_BANDS_UM,
    *,
    wavelength_min_um: float = WAVELENGTH_MIN_UM,
    wavelength_max_um: float = WAVELENGTH_MAX_UM,
) -> dict[str, tuple[float, float]]:
    """Clip named bands to the configured retained wavelength interval."""
    if not 0 < wavelength_min_um < wavelength_max_um:
        raise ValueError("Wavelength limits must satisfy 0 < min < max.")
    result: dict[str, tuple[float, float]] = {}
    for name, (lower, upper) in bands_um.items():
        lower = max(float(lower), wavelength_min_um)
        upper = min(float(upper), wavelength_max_um)
        if upper > lower:
            result[str(name)] = (lower, upper)
    if not result:
        raise ValueError("No wavelength bands overlap the retained interval.")
    return result


def psd_band_metrics(
    height_um: np.ndarray,
    spacing_um: float,
    *,
    bands_um: Mapping[str, tuple[float, float]] = DEFAULT_WAVELENGTH_BANDS_UM,
) -> dict[str, float | int]:
    """Exact discrete 2D power, RMS, and power fraction in named bands."""
    bands = validated_wavelength_bands(bands_um)
    leveled = detrend_surface(np.asarray(height_um, dtype=float), spacing_um, order=1)
    f0, f1, psd2d, parseval_error = periodogram_2d(leveled, spacing_um)
    F1, F0 = np.meshgrid(f1, f0)
    frequency = np.sqrt(F0**2 + F1**2)
    wavelength = np.full_like(frequency, np.inf)
    positive = frequency > 0
    wavelength[positive] = 1.0 / frequency[positive]
    df0 = 1.0 / (height_um.shape[0] * spacing_um)
    df1 = 1.0 / (height_um.shape[1] * spacing_um)
    retained = (
        positive
        & (wavelength >= WAVELENGTH_MIN_UM)
        & (wavelength <= WAVELENGTH_MAX_UM)
        & np.isfinite(psd2d)
    )
    total_power = float(np.sum(psd2d[retained]) * df0 * df1)
    out: dict[str, float | int] = {
        "retained_psd_power_um2": total_power,
        "retained_psd_rms_um": np.sqrt(total_power) if total_power >= 0 else np.nan,
        "parseval_relative_error": parseval_error,
    }
    band_items = list(bands.items())
    for index, (name, (lower, upper)) in enumerate(band_items):
        upper_comparison = wavelength <= upper if index == len(band_items) - 1 else wavelength < upper
        mask = retained & (wavelength >= lower) & upper_comparison
        power = float(np.sum(psd2d[mask]) * df0 * df1)
        out[f"psd_power_{name}_um2"] = power
        out[f"psd_rms_{name}_um"] = np.sqrt(power) if power >= 0 else np.nan
        out[f"psd_fraction_{name}"] = power / total_power if total_power > 0 else np.nan
        out[f"psd_modes_{name}"] = int(np.count_nonzero(mask))
    return out


def experimental_psd_band_table(
    roughness_table: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    domain: str = "analysis_crop",
    bands_um: Mapping[str, tuple[float, float]] = DEFAULT_WAVELENGTH_BANDS_UM,
) -> pd.DataFrame:
    """Preserve band-power work using the same PSD formula and one crop choice."""
    if domain != "analysis_crop":
        raise ValueError("Experimental PSD bands must use domain='analysis_crop'.")
    rows: list[dict[str, object]] = []
    for record in roughness_table.itertuples(index=False):
        height = np.asarray(exp_heights[record.height_key], dtype=float)
        height = experimental_analysis_crop(height)
        metrics = psd_band_metrics(
            height,
            float(record.spacing_um),
            bands_um=bands_um,
        )
        rows.append({**record._asdict(), "domain": domain, **metrics})
    table = pd.DataFrame(rows)
    group_keys = ["load_mpa", "sample_type", "sample_id"]
    for name in validated_wavelength_bands(bands_um):
        power_column = f"psd_power_{name}_um2"
        rms_column = f"psd_rms_{name}_um"
        initial_power = table.groupby(group_keys)[power_column].transform("first")
        initial_rms = table.groupby(group_keys)[rms_column].transform("first")
        table[f"psd_gain_{name}"] = table[power_column] / initial_power.where(
            initial_power > 0
        )
        table[f"delta_psd_rms_{name}_um"] = table[rms_column] - initial_rms
    return table


def nonsectioned_spatial_tables(
    frame: pd.DataFrame,
    heights: Mapping[str, np.ndarray],
    *,
    source: str,
    domains: Sequence[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Whole-crop experimental or whole-face simulation PSD/ACF, without tiling."""
    if source not in {"exp", "sim"}:
        raise ValueError("source must be 'exp' or 'sim'.")
    allowed_domains = {"analysis_crop"} if source == "exp" else {"full_face"}
    if domains is None:
        domains = tuple(allowed_domains)
    invalid_domains = set(domains) - allowed_domains
    if invalid_domains:
        raise ValueError(f"Unknown non-sectioned domains: {sorted(invalid_domains)}")
    curve_rows: list[dict[str, object]] = []
    scalar_rows: list[dict[str, object]] = []
    for record in frame.itertuples(index=False):
        raw = np.asarray(heights[record.height_key], dtype=float)
        for domain in domains:
            selected_height = (
                experimental_analysis_crop(raw) if source == "exp" else raw
            )
            curves = nonsectioned_spatial_curves(
                selected_height,
                float(record.spacing_um),
            )
            metadata = {
                "source": source,
                "domain": domain,
                "load_mpa": int(record.load_mpa),
                "sample_type": record.sample_type,
                "replicate_id": record.replicate_id,
                "sample_id": getattr(record, "sample_id", record.replicate_id),
                "micro_id": getattr(record, "micro_id", ""),
                "face_index": getattr(record, "face_index", np.nan),
                "unique_plane": bool(getattr(record, "unique_plane", True)),
                "time_h": float(record.time_h),
                "bulk_z_strain_percent": float(record.bulk_z_strain_percent),
                "strain_group": getattr(record, "strain_group", pd.NA),
                "paired_eligible": bool(
                    getattr(record, "primary_paired_spatial_eligible", True)
                ),
                "is_initial": bool(record.is_initial),
                "is_endpoint": bool(record.is_endpoint),
            }
            for curve_type, x_key, y_key in (
                ("psd_normalized", "wavelength_um", "normalized_psd_um_inv"),
                ("psd_absolute", "wavelength_um", "radial_psd_um4"),
                ("acf", "acf_lag_um", "acf"),
            ):
                is_psd = curve_type.startswith("psd_")
                _append_curve(
                    curve_rows,
                    metadata,
                    curve_type=curve_type,
                    x=np.asarray(curves[x_key]),
                    y=np.asarray(curves[y_key]),
                    x_lower=(
                        np.asarray(curves["wavelength_lower_um"])
                        if is_psd
                        else None
                    ),
                    x_upper=(
                        np.asarray(curves["wavelength_upper_um"])
                        if is_psd
                        else None
                    ),
                )
            scalar_rows.append(
                {
                    **metadata,
                    "acf_one_over_e_um": curves["acf_one_over_e_um"],
                    "acf_first_zero_um": curves["acf_first_zero_um"],
                    "acf_one_over_e_crossed": curves["acf_one_over_e_crossed"],
                    "acf_first_zero_crossed": curves["acf_first_zero_crossed"],
                    "spectral_median_wavelength_um": curves[
                        "spectral_median_wavelength_um"
                    ],
                    "long_wavelength_power_fraction": curves[
                        "long_wavelength_power_fraction"
                    ],
                    "high_frequency_exponent": curves["high_frequency_exponent"],
                }
            )
    return pd.DataFrame(curve_rows), pd.DataFrame(scalar_rows)


def summarize_nonsectioned_endpoint_curves(
    curves: pd.DataFrame,
    *,
    experimental_domain: str = "analysis_crop",
    simulation_domain: str = "full_face",
) -> pd.DataFrame:
    """Endpoint hierarchy comparing whole experimental crop with whole sim face."""
    group_keys = [
        "source",
        "comparison_domain",
        "load_mpa",
        "sample_type",
        "curve_type",
        "x_um",
        "x_lower_um",
        "x_upper_um",
    ]
    required = (set(group_keys) - {"comparison_domain"}) | {
        "is_endpoint",
        "domain",
        "y",
        "unique_plane",
        "micro_id",
    }
    if curves.empty or not required.issubset(curves.columns):
        return pd.DataFrame(columns=[*group_keys, "mean", "median", "std", "n"])
    selected = curves[
        curves["is_endpoint"]
        & (
            ((curves["source"] == "exp") & (curves["domain"] == experimental_domain))
            | ((curves["source"] == "sim") & (curves["domain"] == simulation_domain))
        )
    ].copy()
    selected["comparison_domain"] = (
        f"{experimental_domain}_vs_{simulation_domain}"
    )
    exp = selected[selected["source"] == "exp"]
    exp_summary = (
        exp.groupby(group_keys, as_index=False)["y"]
        .agg(mean="mean", median="median", std="std", n="count")
    )
    sim = selected[
        (selected["source"] == "sim")
        & selected["unique_plane"]
        & selected["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
    ]
    sim_realization = (
        sim.groupby([*group_keys, "micro_id"], as_index=False)["y"].mean()
    )
    sim_summary = (
        sim_realization.groupby(group_keys, as_index=False)["y"]
        .agg(mean="mean", median="median", std="std", n="count")
    )
    return pd.concat([exp_summary, sim_summary], ignore_index=True)


def height_distribution_curves(
    grouped_table: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    domain: str = "analysis_crop",
    standardized: bool = False,
    n_bins: int = 100,
) -> pd.DataFrame:
    """Map-balanced height densities summarized by viridis strain groups."""
    if "strain_group" not in grouped_table:
        raise ValueError("Call assign_strain_groups before height distributions.")
    if domain != "analysis_crop":
        raise ValueError("Experimental distributions require analysis_crop.")
    prepared: list[tuple[object, np.ndarray]] = []
    robust_limits: list[tuple[float, float]] = []
    for record in grouped_table.itertuples(index=False):
        height = np.asarray(exp_heights[record.height_key], dtype=float)
        height = experimental_leveled_height(height, float(record.spacing_um))
        height -= np.mean(height)
        if standardized:
            scale = np.sqrt(np.mean(height**2))
            if scale <= 0:
                continue
            height = height / scale
        values = height[np.isfinite(height)]
        if values.size:
            prepared.append((record, values))
            robust_limits.append(tuple(np.percentile(values, [0.2, 99.8])))
    if not prepared:
        return pd.DataFrame()
    lower = float(min(limit[0] for limit in robust_limits))
    upper = float(max(limit[1] for limit in robust_limits))
    edges = np.linspace(lower, upper, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    rows: list[dict[str, object]] = []
    for record, values in prepared:
        density, _ = np.histogram(values, bins=edges, density=True)
        for center, value in zip(centers, density):
            rows.append(
                {
                    "load_mpa": int(record.load_mpa),
                    "sample_type": record.sample_type,
                    "sample_id": record.sample_id,
                    "time_h": float(record.time_h),
                    "bulk_z_strain_percent": float(record.bulk_z_strain_percent),
                    "strain_group": record.strain_group,
                    "domain": domain,
                    "standardized": standardized,
                    "height": float(center),
                    "density": float(value),
                }
            )
    return pd.DataFrame(rows)


def local_sa_profile_z(
    height_um: np.ndarray,
    spacing_um: float,
    *,
    n_bins: int = 40,
) -> pd.DataFrame:
    """Local axial Sa/Sq profile retained from the original analysis."""
    height = np.asarray(height_um, dtype=float)
    indices = np.array_split(np.arange(height.shape[0]), n_bins)
    rows = []
    for index in indices:
        if index.size == 0:
            continue
        block = height[index]
        block = block - np.nanmean(block)
        center_um = float(np.mean(index) * spacing_um)
        rows.append(
            {
                "z_center_um": center_um,
                "z_norm": center_um / max((height.shape[0] - 1) * spacing_um, 1.0),
                "local_sa_um": float(np.nanmean(np.abs(block))),
                "local_sq_um": float(np.sqrt(np.nanmean(block**2))),
                "n_pixels": int(np.count_nonzero(np.isfinite(block))),
            }
        )
    return pd.DataFrame(rows)


def incremental_roughening_rates(table: pd.DataFrame) -> pd.DataFrame:
    """Sequential finite differences in acquisition-time order."""
    rows = []
    for (load, sample_type, sample_id), group in table.groupby(
        ["load_mpa", "sample_type", "sample_id"]
    ):
        group = group.sort_values("time_h")
        strain = group["bulk_z_strain_percent"].to_numpy(dtype=float)
        roughness = group["sa_um"].to_numpy(dtype=float)
        time_h = group["time_h"].to_numpy(dtype=float)
        for index in range(1, len(group)):
            delta_strain = strain[index] - strain[index - 1]
            valid_increment = bool(np.isfinite(delta_strain) and delta_strain > 0)
            delta_sa = roughness[index] - roughness[index - 1]
            rows.append(
                {
                    "load_mpa": int(load),
                    "sample_type": sample_type,
                    "sample_id": sample_id,
                    "time_mid_h": 0.5 * (time_h[index] + time_h[index - 1]),
                    "strain_mid_percent": 0.5 * (strain[index] + strain[index - 1]),
                    "delta_strain_percent": delta_strain,
                    "delta_sa_um": delta_sa,
                    "valid_positive_strain_increment": valid_increment,
                    "roughening_rate_um_per_percent": (
                        delta_sa / delta_strain if valid_increment else np.nan
                    ),
                }
            )
    return pd.DataFrame(rows)


def linear_free(strain_percent: np.ndarray, slope: float, intercept: float) -> np.ndarray:
    """Unconstrained linear comparison model."""
    return slope * np.asarray(strain_percent) + intercept


def linear_through_origin(strain_percent: np.ndarray, slope: float) -> np.ndarray:
    """Baseline-referenced manuscript slope model."""
    return slope * np.asarray(strain_percent)


def threshold_linear(
    strain_percent: np.ndarray,
    slope: float,
    threshold_percent: float,
) -> np.ndarray:
    """Genuinely distinct onset model retained from the original analysis."""
    return slope * np.maximum(np.asarray(strain_percent) - threshold_percent, 0.0)


def threshold_power(
    strain_percent: np.ndarray,
    amplitude: float,
    threshold_percent: float,
    exponent: float,
) -> np.ndarray:
    """Genuinely distinct post-threshold power-law model."""
    return amplitude * np.maximum(
        np.asarray(strain_percent) - threshold_percent,
        0.0,
    ) ** exponent


def fit_roughening_models(table: pd.DataFrame) -> pd.DataFrame:
    """Fit genuinely distinct models separately by load, weighting specimens equally."""
    specifications = (
        ("linear_free", linear_free, (0.1, 0.0), (-np.inf, np.inf)),
        ("linear_origin", linear_through_origin, (0.1,), (0.0, np.inf)),
    )
    rows = []
    for (load, sample_type), case in table.groupby(["load_mpa", "sample_type"]):
        data = case[
            np.isfinite(case["bulk_z_strain_percent"])
            & np.isfinite(case["delta_sa_um"])
        ].copy()
        if len(data) < 4 or data["sample_id"].nunique() < 2:
            continue
        x = data["bulk_z_strain_percent"].to_numpy(dtype=float)
        y = data["delta_sa_um"].to_numpy(dtype=float)
        observations_per_specimen = data.groupby("sample_id")["sample_id"].transform(
            "size"
        )
        sigma = np.sqrt(observations_per_specimen.to_numpy(dtype=float))
        case_specifications = (
            *specifications,
            (
                "threshold_linear",
                threshold_linear,
                (0.1, 0.0),
                ([0.0, 0.0], [np.inf, np.max(x)]),
            ),
            (
                "threshold_power",
                threshold_power,
                (0.1, 0.0, 1.0),
                ([0.0, 0.0, 0.05], [np.inf, np.max(x), 5.0]),
            ),
        )
        for name, function, initial, bounds in case_specifications:
            try:
                parameters, covariance = curve_fit(
                    function,
                    x,
                    y,
                    p0=initial,
                    bounds=bounds,
                    sigma=sigma,
                    absolute_sigma=False,
                    maxfev=50_000,
                )
                residual = y - function(x, *parameters)
                rows.append(
                    {
                        "load_mpa": int(load),
                        "sample_type": sample_type,
                        "model": name,
                        "parameters": parameters,
                        "parameter_se": np.sqrt(np.diag(covariance)),
                        "rmse_um": float(np.sqrt(np.mean(residual**2))),
                        "n": int(x.size),
                        "n_specimens": int(data["sample_id"].nunique()),
                        "weighting": "equal total weight per specimen",
                    }
                )
            except Exception as exc:
                warnings.warn(f"Could not fit {load} {sample_type} {name}: {exc}")
    return pd.DataFrame(rows)




## Spatial curve tables


In [ ]:
def _append_curve(
    rows: list[dict[str, object]],
    metadata: Mapping[str, object],
    *,
    curve_type: str,
    x: np.ndarray,
    y: np.ndarray,
    x_lower: np.ndarray | None = None,
    x_upper: np.ndarray | None = None,
) -> None:
    x = np.asarray(x)
    y = np.asarray(y)
    if x_lower is None:
        x_lower = x
    if x_upper is None:
        x_upper = x
    for x_value, lower, upper, y_value in zip(x, x_lower, x_upper, y):
        if np.isfinite(x_value) and np.isfinite(y_value):
            rows.append(
                {
                    **metadata,
                    "curve_type": curve_type,
                    "x_um": float(x_value),
                    "x_lower_um": float(lower),
                    "x_upper_um": float(upper),
                    "y": float(y_value),
                }
            )


def endpoint_spatial_curves(
    exp_df: pd.DataFrame,
    sim_df: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    sim_heights: Mapping[str, np.ndarray],
    *,
    include_588_descriptive: bool = True,
    include_legacy_diagnostic: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate endpoint curves without silently imposing a paired cohort."""
    curve_rows: list[dict[str, object]] = []
    scalar_rows: list[dict[str, object]] = []

    exp_endpoint = exp_df[exp_df["is_endpoint"]].copy()
    if not include_588_descriptive:
        exp_endpoint = exp_endpoint[exp_endpoint["load_mpa"] != 588]
    for record in exp_endpoint.itertuples(index=False):
        curves = map_spatial_curves(
            np.asarray(exp_heights[record.height_key], dtype=float),
            float(record.spacing_um),
            source="exp",
        )
        metadata = {
            "source": "exp",
            "load_mpa": int(record.load_mpa),
            "sample_type": record.sample_type,
            "replicate_id": record.sample_id,
            "micro_id": "",
            "face_index": np.nan,
            "descriptive_only": int(record.load_mpa) == 588,
            "time_h": float(record.time_h),
            "bulk_z_strain_percent": float(record.bulk_z_strain_percent),
        }
        curve_specifications = [
            ("psd_normalized", "wavelength_um", "normalized_psd_um_inv"),
            ("psd_absolute", "wavelength_um", "radial_psd_um4"),
            ("acf", "acf_lag_um", "acf"),
        ]
        if include_legacy_diagnostic:
            curve_specifications.append(
                ("psd_legacy_diagnostic", "wavelength_um", "legacy_normalized_psd_um")
            )
        for curve_type, x_key, y_key in curve_specifications:
            is_psd = curve_type.startswith("psd_")
            _append_curve(
                curve_rows,
                metadata,
                curve_type=curve_type,
                x=np.asarray(curves[x_key]),
                y=np.asarray(curves[y_key]),
                x_lower=(
                    np.asarray(curves["wavelength_lower_um"]) if is_psd else None
                ),
                x_upper=(
                    np.asarray(curves["wavelength_upper_um"]) if is_psd else None
                ),
            )
        scalar_rows.append(
            {
                **metadata,
                "acf_one_over_e_um": curves["acf_one_over_e_um"],
                "acf_first_zero_um": curves["acf_first_zero_um"],
                "acf_one_over_e_crossed_fraction": curves[
                    "acf_one_over_e_crossed_fraction"
                ],
                "acf_first_zero_crossed_fraction": curves[
                    "acf_first_zero_crossed_fraction"
                ],
                "spectral_median_wavelength_um": curves[
                    "spectral_median_wavelength_um"
                ],
                "long_wavelength_power_fraction": curves[
                    "long_wavelength_power_fraction"
                ],
                "high_frequency_exponent": curves["high_frequency_exponent"],
                "n_windows": curves["n_windows"],
            }
        )

    sim_endpoint = sim_df[
        sim_df["is_endpoint"]
        & sim_df["unique_plane"]
        & sim_df["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
    ].copy()
    for record in sim_endpoint.itertuples(index=False):
        curves = map_spatial_curves(
            np.asarray(sim_heights[record.height_key], dtype=float),
            float(record.spacing_um),
            source="sim",
        )
        metadata = {
            "source": "sim",
            "load_mpa": int(record.load_mpa),
            "sample_type": record.sample_type,
            "replicate_id": record.micro_id,
            "micro_id": record.micro_id,
            "face_index": int(record.face_index),
            "descriptive_only": False,
            "time_h": float(record.time_h),
            "bulk_z_strain_percent": float(record.bulk_z_strain_percent),
        }
        curve_specifications = [
            ("psd_normalized", "wavelength_um", "normalized_psd_um_inv"),
            ("psd_absolute", "wavelength_um", "radial_psd_um4"),
            ("acf", "acf_lag_um", "acf"),
        ]
        if include_legacy_diagnostic:
            curve_specifications.append(
                ("psd_legacy_diagnostic", "wavelength_um", "legacy_normalized_psd_um")
            )
        for curve_type, x_key, y_key in curve_specifications:
            is_psd = curve_type.startswith("psd_")
            _append_curve(
                curve_rows,
                metadata,
                curve_type=curve_type,
                x=np.asarray(curves[x_key]),
                y=np.asarray(curves[y_key]),
                x_lower=(
                    np.asarray(curves["wavelength_lower_um"]) if is_psd else None
                ),
                x_upper=(
                    np.asarray(curves["wavelength_upper_um"]) if is_psd else None
                ),
            )
        scalar_rows.append(
            {
                **metadata,
                "acf_one_over_e_um": curves["acf_one_over_e_um"],
                "acf_first_zero_um": curves["acf_first_zero_um"],
                "acf_one_over_e_crossed_fraction": curves[
                    "acf_one_over_e_crossed_fraction"
                ],
                "acf_first_zero_crossed_fraction": curves[
                    "acf_first_zero_crossed_fraction"
                ],
                "spectral_median_wavelength_um": curves[
                    "spectral_median_wavelength_um"
                ],
                "long_wavelength_power_fraction": curves[
                    "long_wavelength_power_fraction"
                ],
                "high_frequency_exponent": curves["high_frequency_exponent"],
                "n_windows": curves["n_windows"],
            }
        )
    return pd.DataFrame(curve_rows), pd.DataFrame(scalar_rows)


def summarize_endpoint_curves(curves: pd.DataFrame) -> pd.DataFrame:
    """Apply the source-specific replicate hierarchy before plotting."""
    exp = curves[curves["source"] == "exp"].copy()
    exp_summary = (
        exp.groupby(
            [
                "source",
                "load_mpa",
                "sample_type",
                "curve_type",
                "x_um",
                "x_lower_um",
                "x_upper_um",
            ],
            as_index=False,
        )["y"]
        .agg(
            mean="mean",
            median="median",
            q25=lambda values: float(np.nanquantile(values, 0.25)),
            q75=lambda values: float(np.nanquantile(values, 0.75)),
            std="std",
            n="count",
        )
    )

    sim = curves[curves["source"] == "sim"].copy()
    # First average the two unique orthogonal faces within each realization.
    sim_realizations = (
        sim.groupby(
            [
                "source",
                "micro_id",
                "load_mpa",
                "sample_type",
                "curve_type",
                "x_um",
                "x_lower_um",
                "x_upper_um",
            ],
            as_index=False,
        )["y"]
        .mean()
    )
    # Then average the realization-level curves.
    sim_summary = (
        sim_realizations.groupby(
            [
                "source",
                "load_mpa",
                "sample_type",
                "curve_type",
                "x_um",
                "x_lower_um",
                "x_upper_um",
            ],
            as_index=False,
        )["y"]
        .agg(
            mean="mean",
            median="median",
            q25=lambda values: float(np.nanquantile(values, 0.25)),
            q75=lambda values: float(np.nanquantile(values, 0.75)),
            std="std",
            n="count",
        )
    )
    return pd.concat([exp_summary, sim_summary], ignore_index=True)


def summarize_endpoint_scalars(scalars: pd.DataFrame) -> pd.DataFrame:
    """Summarize scalar descriptors with experiment/simulation replicate hierarchy."""
    value_columns = [
        "acf_one_over_e_um",
        "acf_first_zero_um",
        "acf_one_over_e_crossed_fraction",
        "acf_first_zero_crossed_fraction",
        "spectral_median_wavelength_um",
        "long_wavelength_power_fraction",
        "high_frequency_exponent",
        "n_windows",
    ]
    exp = scalars[scalars["source"] == "exp"].copy()
    exp_summary = (
        exp.groupby(["source", "load_mpa", "sample_type"], as_index=False)[
            value_columns
        ]
        .agg(["mean", "median", "std", "count"])
    )
    exp_summary.columns = [
        "_".join(part for part in column if part)
        if isinstance(column, tuple)
        else column
        for column in exp_summary.columns
    ]

    sim = scalars[scalars["source"] == "sim"].copy()
    sim_realizations = (
        sim.groupby(
            ["source", "load_mpa", "sample_type", "micro_id"],
            as_index=False,
        )[value_columns]
        .mean()
    )
    sim_summary = (
        sim_realizations.groupby(
            ["source", "load_mpa", "sample_type"], as_index=False
        )[value_columns]
        .agg(["mean", "median", "std", "count"])
    )
    sim_summary.columns = [
        "_".join(part for part in column if part)
        if isinstance(column, tuple)
        else column
        for column in sim_summary.columns
    ]
    return pd.concat([exp_summary, sim_summary], ignore_index=True)


def summarize_paired_curves(curves: pd.DataFrame) -> pd.DataFrame:
    """Median and interquartile paired curves for manuscript Figure 9."""
    if curves.empty:
        return pd.DataFrame()
    return (
        curves.groupby(["load_mpa", "sample_type", "curve_type", "x_um"])[
            "y"
        ]
        .agg(
            median="median",
            q25=lambda values: float(np.nanquantile(values, 0.25)),
            q75=lambda values: float(np.nanquantile(values, 0.75)),
            n="count",
        )
        .reset_index()
    )


def paired_spatial_evolution(
    exp_df: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    loads: Sequence[int] = (500, 530),
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Specimen-paired raw PSD gain and ACF change for manuscript Figure 9."""
    curve_rows: list[dict[str, object]] = []
    scalar_rows: list[dict[str, object]] = []
    eligible = exp_df[
        exp_df["load_mpa"].isin(loads)
        & exp_df["primary_paired_spatial_eligible"]
    ]
    for (load_mpa, sample_type, sample_id), group in eligible.groupby(
        ["load_mpa", "sample_type", "sample_id"]
    ):
        group = group.sort_values("time_h")
        if len(group) < 2:
            continue
        initial_record = group.iloc[0]
        final_record = group.iloc[-1]
        initial = map_spatial_curves(
            np.asarray(exp_heights[initial_record["height_key"]], dtype=float),
            float(initial_record["spacing_um"]),
            source="exp",
        )
        final = map_spatial_curves(
            np.asarray(exp_heights[final_record["height_key"]], dtype=float),
            float(final_record["spacing_um"]),
            source="exp",
        )
        raw_initial = np.asarray(initial["radial_psd_um4"])
        raw_final = np.asarray(final["radial_psd_um4"])
        gain = np.full_like(raw_initial, np.nan)
        valid = (
            np.isfinite(raw_initial)
            & np.isfinite(raw_final)
            & (raw_initial > 0)
            & (raw_final > 0)
        )
        gain[valid] = np.log10(raw_final[valid] / raw_initial[valid])
        acf_change = np.asarray(final["acf"]) - np.asarray(initial["acf"])
        metadata = {
            "load_mpa": int(load_mpa),
            "sample_type": sample_type,
            "sample_id": sample_id,
        }
        _append_curve(
            curve_rows,
            metadata,
            curve_type="psd_gain_decades",
            x=np.asarray(initial["wavelength_um"]),
            y=gain,
        )
        _append_curve(
            curve_rows,
            metadata,
            curve_type="acf_change",
            x=np.asarray(initial["acf_lag_um"]),
            y=acf_change,
        )
        scalar_rows.append(
            {
                **metadata,
                "acf_one_over_e_initial_um": initial["acf_one_over_e_um"],
                "acf_one_over_e_final_um": final["acf_one_over_e_um"],
                "acf_one_over_e_change_um": (
                    float(final["acf_one_over_e_um"])
                    - float(initial["acf_one_over_e_um"])
                ),
            }
        )
    return pd.DataFrame(curve_rows), pd.DataFrame(scalar_rows)


def paired_nonsectioned_spatial_evolution(
    curves: pd.DataFrame,
) -> pd.DataFrame:
    """Derive paired whole-crop PSD gain and ACF change from cached curves."""
    required = {
        "source",
        "domain",
        "load_mpa",
        "sample_type",
        "sample_id",
        "curve_type",
        "x_um",
        "y",
        "is_initial",
        "is_endpoint",
    }
    if not required.issubset(curves.columns):
        raise ValueError(f"Curve table is missing {sorted(required - set(curves))}.")
    selected = curves[
        (curves["source"] == "exp") & curves["paired_eligible"]
    ].copy()
    initial = selected[selected["is_initial"]].copy()
    final = selected[selected["is_endpoint"]].copy()
    keys = [
        "domain",
        "load_mpa",
        "sample_type",
        "sample_id",
        "curve_type",
        "x_um",
    ]
    paired = initial[keys + ["y"]].merge(
        final[keys + ["y"]],
        on=keys,
        suffixes=("_initial", "_final"),
        validate="one_to_one",
    )
    psd = paired[
        (paired["curve_type"] == "psd_absolute")
        & (paired["y_initial"] > 0)
        & (paired["y_final"] > 0)
    ].copy()
    psd["change_type"] = "psd_gain_decades"
    psd["change"] = np.log10(psd["y_final"] / psd["y_initial"])
    acf = paired[paired["curve_type"] == "acf"].copy()
    acf["change_type"] = "acf_change"
    acf["change"] = acf["y_final"] - acf["y_initial"]
    return pd.concat([psd, acf], ignore_index=True)


def delta_h_unregistered_sensitivity(
    initial_raw_height_um: np.ndarray,
    final_raw_height_um: np.ndarray,
    spacing_um: float,
) -> np.ndarray:
    """Crop-first signed experimental final-minus-initial sensitivity."""
    initial_height_um = experimental_analysis_crop(initial_raw_height_um)
    final_height_um = experimental_analysis_crop(final_raw_height_um)
    if initial_height_um.shape != final_height_um.shape:
        raise ValueError("Unregistered delta-h maps must have identical shapes.")
    initial = detrend_surface(initial_height_um, spacing_um, order=1)
    final = detrend_surface(final_height_um, spacing_um, order=1)
    return (final - np.mean(final)) - (initial - np.mean(initial))




## DIC loading and the one retained strain-field figure per dataset


In [ ]:
DIC_CROPS = (
    (6, 6, 9, 5),
    (7, 5, 6, 6),
    (9, 5, 7, 11),
    (7, 3, 7, 7),
)


def _blank_csv_line(line_text: str) -> bool:
    if not line_text.strip():
        return True
    return all(
        cell.strip().strip('"') == ""
        for cell in line_text.rstrip("\n\r").split(",")
    )


def read_dic_blocks(
    path: str | Path,
    *,
    start_row: int = 1086,
    n_datasets: int = 4,
) -> list[pd.DataFrame]:
    """Read the DIC export once and return its blank-line-separated blocks."""
    with Path(path).open("r", newline="") as stream:
        lines = stream.readlines()[start_row - 1 :]
    blocks: list[list[str]] = []
    current: list[str] = []
    for line_text in lines:
        if _blank_csv_line(line_text):
            if current:
                blocks.append(current)
                current = []
                if len(blocks) == n_datasets:
                    break
        else:
            current.append(line_text)
    if current and len(blocks) < n_datasets:
        blocks.append(current)
    datasets = []
    for block in blocks:
        frame = pd.read_csv(StringIO("".join(block)), skipinitialspace=True)
        frame.columns = frame.columns.str.strip().str.strip('"').str.strip()
        datasets.append(frame)
    if len(datasets) != n_datasets:
        raise ValueError(f"Expected {n_datasets} DIC datasets; found {len(datasets)}.")
    return datasets


def _crop_slice(low: int, high: int) -> slice:
    return slice(low, None if high == 0 else -high)


def _fill_invalid_nearest(values: np.ndarray, invalid: np.ndarray) -> np.ndarray:
    if not np.any(invalid):
        return np.asarray(values, dtype=float)
    if np.all(invalid):
        raise ValueError("DIC crop contains no valid displacement points.")
    nearest = distance_transform_edt(
        invalid,
        return_distances=False,
        return_indices=True,
    )
    return np.asarray(values, dtype=float)[tuple(nearest)]


def dic_strain_fields(
    frame: pd.DataFrame,
    crop: tuple[int, int, int, int],
) -> dict[str, np.ndarray]:
    """Calculate axial, transverse, and engineering shear strain on one crop."""
    required = {"x_c", "y_c", "u_c", "v_c", "sigma"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"DIC data are missing columns: {sorted(missing)}")
    if frame.duplicated(["x_c", "y_c"]).any():
        raise ValueError("DIC data contain duplicate (x_c, y_c) coordinates.")
    ordered = frame.sort_values(["y_c", "x_c"]).copy()
    z_values = np.sort(ordered["x_c"].unique())
    y_values = np.sort(ordered["y_c"].unique())
    shape = (y_values.size, z_values.size)
    if len(ordered) != shape[0] * shape[1]:
        raise ValueError("DIC data are not a complete rectangular grid.")
    observed = pd.MultiIndex.from_frame(ordered[["y_c", "x_c"]])
    expected = pd.MultiIndex.from_product(
        [y_values, z_values], names=["y_c", "x_c"]
    )
    if not observed.equals(expected):
        raise ValueError("DIC coordinates do not cover the complete Cartesian grid.")
    Y = ordered["y_c"].to_numpy(dtype=float).reshape(shape)
    Z = ordered["x_c"].to_numpy(dtype=float).reshape(shape)
    U = ordered["u_c"].to_numpy(dtype=float).reshape(shape)
    V = ordered["v_c"].to_numpy(dtype=float).reshape(shape)
    quality = ordered["sigma"].to_numpy(dtype=float).reshape(shape)
    low_y, high_y, low_z, high_z = crop
    selection = (_crop_slice(low_y, high_y), _crop_slice(low_z, high_z))
    Y, Z, U, V, quality = (
        array[selection] for array in (Y, Z, U, V, quality)
    )
    invalid = (~np.isfinite(quality)) | np.isclose(quality, -1.0)
    invalid |= ~np.isfinite(U) | ~np.isfinite(V)
    U_filled = _fill_invalid_nearest(U, invalid)
    V_filled = _fill_invalid_nearest(V, invalid)
    y = Y[:, 0]
    z = Z[0, :]
    dU_dy, dU_dz = np.gradient(U_filled, y, z, edge_order=2)
    dV_dy, dV_dz = np.gradient(V_filled, y, z, edge_order=2)
    invalid_gradient = binary_dilation(invalid, structure=np.ones((3, 3), dtype=bool))
    fields = {
        "Y": Y,
        "Z": Z,
        "quality": np.where(invalid, np.nan, quality),
        "eps_zz": dU_dz,
        "eps_yy": dV_dy,
        "gamma_yz": dU_dy + dV_dz,
    }
    for key in ("eps_zz", "eps_yy", "gamma_yz"):
        fields[key] = np.where(invalid_gradient, np.nan, fields[key])
    fields["invalid_gradient"] = invalid_gradient
    return fields


def dic_axial_strain_profile(fields: Mapping[str, np.ndarray]) -> pd.DataFrame:
    """Transverse mean axial strain as a function of the measured z coordinate."""
    z = np.asarray(fields["Z"], dtype=float)[0]
    axial = np.nanmean(np.asarray(fields["eps_zz"], dtype=float), axis=0)
    valid = np.isfinite(z) & np.isfinite(axial)
    z = z[valid]
    axial = axial[valid]
    if z.size < 3:
        return pd.DataFrame(columns=["z", "z_norm", "eps_zz_mean"])
    return pd.DataFrame(
        {
            "z": z,
            "z_norm": (z - z.min()) / (z.max() - z.min()),
            "eps_zz_mean": axial,
        }
    )


def dic_axial_row_averaged_psd(
    fields: Mapping[str, np.ndarray],
    *,
    wavelength_min_mm: float = DIC_WAVELENGTH_MIN_MM,
    wavelength_max_mm: float = DIC_WAVELENGTH_MAX_MM,
) -> pd.DataFrame:
    """Mean of per-transverse-row axial-strain spectra.

    This replaces the many slightly different row-wise, positive-only, direct
    DFT, transverse-mean, and circular variants in the original file.  Each
    axial row is linearly detrended and tapered before its one-sided spectrum is
    calculated; spectra are then averaged, preserving transverse-incoherent
    strain structure.  It is not mixed with the surface-height PSD.
    """
    if not 0 < wavelength_min_mm < wavelength_max_mm:
        raise ValueError("DIC wavelength limits must satisfy 0 < min < max.")
    z = np.asarray(fields["Z"], dtype=float)[0]
    axial = np.asarray(fields["eps_zz"], dtype=float)
    if z.size < 4 or axial.ndim != 2 or axial.shape[1] != z.size:
        return pd.DataFrame()
    spacing = float(np.median(np.diff(z)))
    if spacing <= 0 or not np.allclose(np.diff(z), spacing, rtol=1.0e-3, atol=1.0e-12):
        raise ValueError("DIC axial coordinates must be uniformly increasing.")
    design = np.column_stack((np.ones_like(z), z))
    window = np.hanning(z.size)
    row_spectra = []
    for row in axial:
        finite = np.isfinite(row)
        if np.count_nonzero(finite) < max(4, int(0.8 * row.size)):
            continue
        filled = np.interp(z, z[finite], row[finite])
        residual = filled - design @ np.linalg.lstsq(design, filled, rcond=None)[0]
        transform = np.fft.rfft(residual * window)
        spectrum = spacing * np.abs(transform) ** 2 / np.sum(window**2)
        if residual.size % 2 == 0:
            spectrum[1:-1] *= 2.0
        else:
            spectrum[1:] *= 2.0
        row_spectra.append(spectrum)
    if not row_spectra:
        return pd.DataFrame()
    frequency = np.fft.rfftfreq(z.size, d=spacing)
    psd = np.mean(np.vstack(row_spectra), axis=0)
    valid = (
        (frequency > 0)
        & (frequency >= 1.0 / wavelength_max_mm)
        & (frequency <= 1.0 / wavelength_min_mm)
        & np.isfinite(psd)
    )
    return pd.DataFrame(
        {
            "frequency_mm_inv": frequency[valid],
            "wavelength_mm": 1.0 / frequency[valid],
            "axial_strain_psd_mm": psd[valid],
            "n_transverse_rows": len(row_spectra),
        }
    ).sort_values("wavelength_mm")


def correlate_local_sa_with_dic(
    raw_experimental_height_um: np.ndarray,
    height_spacing_um: float,
    fields: Mapping[str, np.ndarray],
    *,
    n_bins: int = 40,
) -> tuple[pd.DataFrame, float]:
    """Optional crop-first local experimental Sa--DIC correlation."""
    height_um = experimental_leveled_height(
        raw_experimental_height_um,
        height_spacing_um,
    )
    roughness = local_sa_profile_z(height_um, height_spacing_um, n_bins=n_bins)
    strain = dic_axial_strain_profile(fields)
    if roughness.empty or strain.empty:
        return pd.DataFrame(), np.nan
    interpolated = np.interp(
        roughness["z_norm"].to_numpy(dtype=float),
        strain["z_norm"].to_numpy(dtype=float),
        strain["eps_zz_mean"].to_numpy(dtype=float),
    )
    result = roughness.copy()
    result["eps_zz_interp"] = interpolated
    valid = np.isfinite(result["local_sa_um"]) & np.isfinite(
        result["eps_zz_interp"]
    )
    correlation = (
        float(
            np.corrcoef(
                result.loc[valid, "local_sa_um"],
                result.loc[valid, "eps_zz_interp"],
            )[0, 1]
        )
        if np.count_nonzero(valid) >= 3
        else np.nan
    )
    return result, correlation


def plot_dic_final_strain_fields(
    dic_datasets: Sequence[pd.DataFrame],
    *,
    crops: Sequence[tuple[int, int, int, int]] = DIC_CROPS,
    save: bool = True,
) -> list[plt.Figure]:
    """Keep exactly one four-panel final-crop/strain figure per DIC dataset."""
    figures = []
    for dataset_index, (frame, crop) in enumerate(zip(dic_datasets, crops)):
        fields = dic_strain_fields(frame, crop)
        strain_values = np.concatenate(
            [
                fields[key][np.isfinite(fields[key])]
                for key in ("eps_zz", "eps_yy", "gamma_yz")
            ]
        )
        limit = float(np.percentile(np.abs(strain_values), 99.0))
        limit = limit if np.isfinite(limit) and limit > 0 else 1.0
        strain_norm = mpl.colors.TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)
        strain_cmap = mpl.colormaps["coolwarm"].copy()
        strain_cmap.set_bad("0.85")
        quality_cmap = mpl.colormaps["viridis"].copy()
        quality_cmap.set_bad("0.85")

        fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.2), constrained_layout=True)
        panels = (
            ("Final crop / quality", "quality", quality_cmap, None),
            (r"Axial strain, $\epsilon_{zz}$", "eps_zz", strain_cmap, strain_norm),
            (r"Transverse strain, $\epsilon_{yy}$", "eps_yy", strain_cmap, strain_norm),
            (r"Shear strain, $\gamma_{yz}$", "gamma_yz", strain_cmap, strain_norm),
        )
        strain_image = None
        for axis, (title, key, cmap, norm) in zip(axes, panels):
            image = axis.pcolormesh(
                fields["Y"],
                fields["Z"],
                fields[key],
                shading="auto",
                cmap=cmap,
                norm=norm,
                rasterized=True,
            )
            axis.set_aspect("equal", adjustable="box")
            axis.set_title(title)
            axis.set_xlabel("transverse position, y")
            axis.set_ylabel("axial position, z")
            if key == "quality":
                fig.colorbar(image, ax=axis, label="DIC quality")
            else:
                strain_image = image
        if strain_image is not None:
            fig.colorbar(
                strain_image,
                ax=list(axes[1:]),
                label="engineering strain",
                shrink=0.86,
            )
        fig.suptitle(f"DIC dataset {dataset_index}: final valid crop and strain fields")
        if save:
            fig.savefig(
                OUTPUT_DIR / f"dic_dataset_{dataset_index}_final_strain_fields.png",
                bbox_inches="tight",
            )
        figures.append(fig)
    return figures




## Shared figure helpers


In [ ]:
def _finish_figure(
    fig: plt.Figure,
    filename: str,
    *,
    save: bool,
) -> plt.Figure:
    if save:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    return fig


def _case_label(load_mpa: int, sample_type: str) -> str:
    mode = "interrupted" if sample_type == "int" else "uninterrupted"
    suffix = ", descriptive" if int(load_mpa) == 588 else ""
    return f"{int(load_mpa)} MPa, {mode}{suffix}"


def _case_axes(
    *,
    sharex: bool = False,
    sharey: bool = False,
) -> tuple[plt.Figure, np.ndarray]:
    return plt.subplots(
        2,
        3,
        figsize=(10.0, 6.0),
        sharex=sharex,
        sharey=sharey,
        constrained_layout=True,
    )


def calibration_parameter_table(
    path: str | Path = CALIBRATION_PARAMS_PATH,
    *,
    loads: Sequence[int] = (500, 530, 588),
) -> pd.DataFrame:
    """Read the selected parameter row once and expose load-dependent values."""
    frame = pd.read_csv(Path(path).expanduser())
    if frame.empty:
        raise ValueError(f"Calibration table {path} is empty.")
    row = frame.iloc[0]
    records = []
    for load in loads:
        nrsx_key = f"nrsx_{int(load)}"
        gamd0x_key = f"gamd0x_{int(load)}"
        if nrsx_key not in row or gamd0x_key not in row:
            raise KeyError(f"Missing {nrsx_key!r} or {gamd0x_key!r} in {path}.")
        nrsx = float(row[nrsx_key])
        records.append(
            {
                "load_mpa": int(load),
                "nrsx": nrsx,
                "rate_sensitivity_m": 1.0 / nrsx,
                "gamd0x_s_inv": float(row[gamd0x_key]),
            }
        )
    return pd.DataFrame(records)




## Mechanical and calibration figures


In [ ]:
def plot_calibration_parameters(
    parameters: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.1), constrained_layout=True)
    for record in parameters.sort_values("load_mpa").itertuples(index=False):
        color = load_color(record.load_mpa)
        axes[0].scatter(record.load_mpa, record.rate_sensitivity_m, color=color)
        axes[1].scatter(record.load_mpa, record.gamd0x_s_inv, color=color)
    axes[0].plot(
        parameters["load_mpa"], parameters["rate_sensitivity_m"], color="0.45"
    )
    axes[1].plot(parameters["load_mpa"], parameters["gamd0x_s_inv"], color="0.45")
    axes[0].set(xlabel="load (MPa)", ylabel=r"rate sensitivity, $m=1/n$")
    axes[1].set(xlabel="load (MPa)", ylabel=r"reference shear rate, $\dot\gamma_0$ (s$^{-1}$)")
    axes[1].set_yscale("log")
    return _finish_figure(fig, "01_calibration_parameters.png", save=save)


def plot_mechanical_strain_histories(
    histories: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    """Experimental and simulated strain histories with one color per case."""
    fig, ax = plt.subplots(figsize=(6.6, 4.2), constrained_layout=True)
    if histories.empty:
        ax.text(0.5, 0.5, "No cached histories", ha="center", va="center")
        return _finish_figure(fig, "02_mechanical_strain_histories.png", save=save)
    for (source, load, sample_type), group in histories.groupby(
        ["source", "load_mpa", "sample_type"]
    ):
        identity = "sample_id" if source == "exp" else "micro_id"
        first = True
        for _, history in group.groupby(identity):
            history = history.sort_values("time_h")
            ax.plot(
                history["time_h"],
                history["bulk_z_strain_percent"],
                color=load_color(load),
                linestyle=SOURCE_LINESTYLES[source],
                alpha=0.45 if source == "exp" else 0.70,
                label=f"{int(load)} MPa {source}" if first else None,
            )
            first = False
    ax.set_xlabel("time (h)")
    ax.set_ylabel(r"bulk axial strain, $\epsilon_{zz}$ (%)")
    ax.legend(ncol=2)
    return _finish_figure(fig, "02_mechanical_strain_histories.png", save=save)




## Height-map and roughness figures


In [ ]:
def plot_raw_map_qc_and_analysis_crop(
    raw_height_um: np.ndarray,
    spacing_um: float,
    *,
    title: str = "Representative experimental height map",
    save: bool = True,
) -> plt.Figure:
    """Show acquisition/QC fields beside the sole experimental analysis field."""
    raw = np.asarray(raw_height_um, dtype=float)
    full = detrend_surface(raw, spacing_um, order=1)
    crop = experimental_leveled_height(raw, spacing_um)
    fields = (raw - np.mean(raw), full - np.mean(full), crop - np.mean(crop))
    limit = float(max(np.percentile(np.abs(field), 99.0) for field in fields))
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.3), constrained_layout=True)
    titles = (
        "Raw acquisition map\nQC display only",
        "Plane-leveled acquisition map\nQC display only",
        f"Analysis crop\n$S_a$={surface_metrics(fields[2])['sa_um']:.3g} µm",
    )
    image = None
    for axis, field, panel_title in zip(axes, fields, titles):
        image = axis.imshow(
            field,
            origin="lower",
            cmap="coolwarm",
            vmin=-limit,
            vmax=limit,
            extent=(
                0,
                field.shape[1] * spacing_um,
                0,
                field.shape[0] * spacing_um,
            ),
            aspect="equal",
            interpolation="nearest",
            rasterized=True,
        )
        axis.set_title(panel_title)
        axis.set_xlabel("transverse position (µm)")
        axis.set_ylabel("loading position (µm)")
    if image is not None:
        fig.colorbar(image, ax=list(axes), label="mean-removed height (µm)", shrink=0.82)
    fig.suptitle(title)
    return _finish_figure(fig, "03_raw_map_qc_and_analysis_crop.png", save=save)


def plot_roughness_case_panels(
    exp_table: pd.DataFrame,
    sim_table: pd.DataFrame,
    *,
    x_column: str = "bulk_z_strain_percent",
    y_column: str = "delta_sa_um",
    save: bool = True,
) -> plt.Figure:
    """Six-case experimental/simulation Sa histories from the unified tables."""
    fig, axes = _case_axes(sharey=True)
    for axis, (load, sample_type) in zip(axes.ravel(), EXPERIMENTS):
        color = load_color(load)
        exp_case = exp_table[
            (exp_table["load_mpa"] == load)
            & (exp_table["sample_type"] == sample_type)
        ]
        for _, specimen in exp_case.groupby("sample_id"):
            specimen = specimen.sort_values(x_column)
            axis.plot(
                specimen[x_column],
                specimen[y_column],
                color=color,
                alpha=0.22,
                linewidth=0.8,
            )
        if not exp_case.empty:
            axis.scatter(
                exp_case[x_column],
                exp_case[y_column],
                color=color,
                marker=TYPE_MARKERS[sample_type],
                s=16,
                label="experiment",
            )

        sim_case = sim_table[
            (sim_table["load_mpa"] == load)
            & (sim_table["sample_type"] == sample_type)
            & sim_table["unique_plane"]
            & sim_table["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
        ]
        if not sim_case.empty:
            realization = (
                sim_case.groupby(["micro_id", x_column], as_index=False)[y_column]
                .mean()
            )
            for _, micro in realization.groupby("micro_id"):
                micro = micro.sort_values(x_column)
                axis.plot(
                    micro[x_column],
                    micro[y_column],
                    color=color,
                    linestyle="--",
                    alpha=0.45,
                    linewidth=1.0,
                )
            axis.scatter(
                realization[x_column],
                realization[y_column],
                facecolor="none",
                edgecolor=color,
                s=20,
                label="simulation",
            )
        axis.axhline(0.0, color="0.65", linewidth=0.7)
        axis.set_title(_case_label(load, sample_type))
        axis.set_xlabel("time (h)" if x_column == "time_h" else r"bulk axial strain, $\epsilon_{zz}$ (%)")
        axis.set_ylabel(r"$\Delta S_a$ (µm)" if y_column == "delta_sa_um" else y_column)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    if handles:
        fig.legend(
            handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=2
        )
    suffix = "time" if x_column == "time_h" else "strain"
    return _finish_figure(fig, f"04_delta_sa_vs_{suffix}_by_case.png", save=save)


def plot_endpoint_delta_sa_by_load(
    exp_table: pd.DataFrame,
    sim_table: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
    exp_endpoint = exp_table[exp_table["is_endpoint"]]
    sim_endpoint = sim_table[
        sim_table["is_endpoint"]
        & sim_table["unique_plane"]
        & sim_table["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
    ]
    sim_realization = (
        sim_endpoint.groupby(
            ["load_mpa", "sample_type", "micro_id"], as_index=False
        )["delta_sa_um"]
        .mean()
    )
    for load, sample_type in EXPERIMENTS:
        color = load_color(load)
        exp_values = exp_endpoint.loc[
            (exp_endpoint["load_mpa"] == load)
            & (exp_endpoint["sample_type"] == sample_type),
            "delta_sa_um",
        ].dropna()
        sim_values = sim_realization.loc[
            (sim_realization["load_mpa"] == load)
            & (sim_realization["sample_type"] == sample_type),
            "delta_sa_um",
        ].dropna()
        for offset, values, marker, label in (
            (-2.0, exp_values, TYPE_MARKERS[sample_type], "experiment"),
            (2.0, sim_values, "D", "simulation"),
        ):
            if values.empty:
                continue
            ax.errorbar(
                load + offset,
                values.mean(),
                yerr=values.std(ddof=1) if len(values) > 1 else None,
                color=color,
                marker=marker,
                markerfacecolor=color if label == "experiment" else "none",
                capsize=2,
                linestyle="none",
                label=label if load == EXPERIMENTS[0][0] else None,
            )
    ax.axhline(0.0, color="0.65", linewidth=0.7)
    ax.set_xlabel("applied stress (MPa)")
    ax.set_ylabel(r"endpoint $\Delta S_a$ (µm)")
    ax.legend()
    return _finish_figure(fig, "06_endpoint_delta_sa_by_load.png", save=save)


def plot_incremental_roughening_rates(
    rates: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
    required = {
        "load_mpa",
        "sample_type",
        "strain_mid_percent",
        "roughening_rate_um_per_percent",
    }
    if rates.empty or not required.issubset(rates.columns):
        ax.text(0.5, 0.5, "No sequential roughening increments", ha="center", va="center")
        ax.set_axis_off()
        return _finish_figure(fig, "07_incremental_roughening_rates.png", save=save)
    for (load, sample_type), group in rates.groupby(["load_mpa", "sample_type"]):
        ax.scatter(
            group["strain_mid_percent"],
            group["roughening_rate_um_per_percent"],
            color=load_color(load),
            marker=TYPE_MARKERS[sample_type],
            label=f"{int(load)} MPa",
            s=20,
        )
    ax.axhline(0.0, color="0.65", linewidth=0.7)
    ax.set_xlabel(r"midpoint bulk axial strain, $\epsilon_{zz}$ (%)")
    ax.set_ylabel(r"finite roughening rate, $\Delta S_a/\Delta\epsilon$ (µm/%)")
    ax.legend(ncol=2)
    return _finish_figure(fig, "07_incremental_roughening_rates.png", save=save)


def plot_roughening_model_comparison(
    table: pd.DataFrame,
    fits: pd.DataFrame,
    *,
    case: tuple[int, str] = (530, "unint"),
    save: bool = True,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
    load, sample_type = case
    selected = table[
        (table["load_mpa"] == load) & (table["sample_type"] == sample_type)
    ]
    for _, group in selected.groupby("sample_id"):
        ax.scatter(
            group["bulk_z_strain_percent"],
            group["delta_sa_um"],
            color=load_color(load),
            marker=TYPE_MARKERS[sample_type],
            alpha=0.55,
            s=18,
        )
    functions = {
        "linear_free": linear_free,
        "linear_origin": linear_through_origin,
        "threshold_linear": threshold_linear,
        "threshold_power": threshold_power,
    }
    styles = ("-", "--", "-.", ":")
    finite_x = selected["bulk_z_strain_percent"].to_numpy(dtype=float)
    finite_x = finite_x[np.isfinite(finite_x)]
    case_fits = fits[
        (fits["load_mpa"] == load) & (fits["sample_type"] == sample_type)
    ]
    if finite_x.size:
        grid = np.linspace(0.0, finite_x.max(), 300)
        for style, record in zip(styles, case_fits.itertuples(index=False)):
            function = functions[record.model]
            parameters = np.asarray(record.parameters, dtype=float)
            ax.plot(
                grid,
                function(grid, *parameters),
                color="black",
                linestyle=style,
                label=f"{record.model}, RMSE={record.rmse_um:.3g} µm",
            )
    ax.axhline(0.0, color="0.65", linewidth=0.7)
    ax.set_xlabel(r"bulk axial strain, $\epsilon_{zz}$ (%)")
    ax.set_ylabel(r"$\Delta S_a$ (µm)")
    ax.set_title(_case_label(*case))
    ax.legend(ncol=2)
    return _finish_figure(fig, "08_roughening_model_comparison.png", save=save)


def plot_surface_parameter_suite(
    table: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    specifications = (
        ("sa_um", r"$S_a$ (µm)"),
        ("sq_um", r"$S_q$ (µm)"),
        ("sz_robust_um", r"robust $S_z$ (µm)"),
        ("ssk", r"$S_{sk}$"),
        ("sku", r"$S_{ku}$"),
        ("ra_anisotropy", r"$R_a^\parallel/R_a^\perp$"),
    )
    fig, axes = plt.subplots(2, 3, figsize=(10.0, 6.0), constrained_layout=True)
    for axis, (column, ylabel) in zip(axes.ravel(), specifications):
        for (load, sample_type), group in table.groupby(["load_mpa", "sample_type"]):
            axis.scatter(
                group["bulk_z_strain_percent"],
                group[column],
                color=load_color(load),
                marker=TYPE_MARKERS[sample_type],
                s=14,
                alpha=0.65,
                label=f"{int(load)} MPa",
            )
        axis.set_xlabel(r"bulk axial strain, $\epsilon_{zz}$ (%)")
        axis.set_ylabel(ylabel)
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=3
    )
    return _finish_figure(fig, "09_surface_parameter_suite.png", save=save)


def _choose_representative_case(
    table: pd.DataFrame,
    *,
    preferred: tuple[int, str] = (530, "unint"),
) -> tuple[int, str]:
    """Use the preferred case when present, otherwise the best sampled case."""
    available = table[
        np.isfinite(table["bulk_z_strain_percent"])
        & table["height_key"].notna()
    ]
    if available.empty:
        raise ValueError("No experimental height records are available for figures.")
    preferred_mask = (
        (available["load_mpa"] == preferred[0])
        & (available["sample_type"] == preferred[1])
    )
    if preferred_mask.any():
        return preferred
    ranking = (
        available.groupby(["load_mpa", "sample_type"], as_index=False)
        .agg(n_maps=("height_key", "size"), n_specimens=("sample_id", "nunique"))
        .sort_values(
            ["n_maps", "n_specimens", "load_mpa", "sample_type"],
            ascending=[False, False, True, True],
        )
    )
    chosen = ranking.iloc[0]
    case = (int(chosen["load_mpa"]), str(chosen["sample_type"]))
    warnings.warn(
        f"Preferred representative case {preferred} is unavailable; using {case}."
    )
    return case


def _representative_records(
    table: pd.DataFrame,
    *,
    case: tuple[int, str] = (530, "unint"),
    n_positive_states: int = 4,
) -> pd.DataFrame:
    """Choose one initial map and nearest positive-strain quantile maps."""
    load, sample_type = case
    selected = table[
        (table["load_mpa"] == load) & (table["sample_type"] == sample_type)
    ].copy()
    selected = selected[np.isfinite(selected["bulk_z_strain_percent"])]
    if selected.empty:
        raise ValueError(f"No records for representative case {case}.")
    specimen_summary = (
        selected.sort_values("time_h")
        .groupby("sample_id", as_index=False)
        .agg(
            n_states=("time_h", "size"),
            strain_min=("bulk_z_strain_percent", "min"),
            strain_max=("bulk_z_strain_percent", "max"),
            endpoint_delta_sa_um=("delta_sa_um", "last"),
        )
    )
    complete = specimen_summary[
        (specimen_summary["n_states"] >= 2)
        & np.isclose(specimen_summary["strain_min"], 0.0, atol=0.05)
    ].copy()
    if complete.empty:
        complete = specimen_summary.copy()
    max_states = int(complete["n_states"].max())
    complete = complete[complete["n_states"] == max_states].copy()
    target_response = float(np.nanmedian(specimen_summary["endpoint_delta_sa_um"]))
    complete["response_distance"] = np.abs(
        complete["endpoint_delta_sa_um"] - target_response
    )
    chosen_sample = str(
        complete.sort_values(["response_distance", "sample_id"]).iloc[0]["sample_id"]
    )
    history = selected[selected["sample_id"].astype(str) == chosen_sample].copy()
    rows = [history.loc[history["bulk_z_strain_percent"].idxmin()]]
    positive = history[history["bulk_z_strain_percent"] > 0]
    if not positive.empty:
        targets = np.unique(
            np.quantile(
                positive["bulk_z_strain_percent"],
                np.linspace(0.0, 1.0, n_positive_states),
            )
        )
        used = {rows[0].name}
        for target in targets:
            distance = (positive["bulk_z_strain_percent"] - target).abs()
            for index in distance.sort_values().index:
                if index not in used:
                    rows.append(history.loc[index])
                    used.add(index)
                    break
    return pd.DataFrame(rows).sort_values("bulk_z_strain_percent")


def _representative_colors(n_records: int) -> list[object]:
    if n_records <= 0:
        return []
    return list(strain_group_colors(n_records))


def plot_representative_surface_maps(
    table: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    case: tuple[int, str] = (530, "unint"),
    n_positive_states: int = 4,
    save: bool = True,
) -> plt.Figure:
    records = _representative_records(
        table, case=case, n_positive_states=n_positive_states
    )
    fields = []
    for record in records.itertuples(index=False):
        height = experimental_leveled_height(
            np.asarray(exp_heights[record.height_key], dtype=float),
            float(record.spacing_um),
        )
        fields.append(height)
    limit = max(float(np.percentile(np.abs(field), 99.0)) for field in fields)
    fig, axes = plt.subplots(
        1,
        len(fields),
        figsize=(2.7 * len(fields), 3.0),
        constrained_layout=True,
        squeeze=False,
    )
    image = None
    for axis, field, record, color in zip(
        axes.ravel(), fields, records.itertuples(index=False), _representative_colors(len(fields))
    ):
        image = axis.imshow(
            field,
            origin="lower",
            cmap="coolwarm",
            vmin=-limit,
            vmax=limit,
            extent=(
                0,
                field.shape[1] * float(record.spacing_um),
                0,
                field.shape[0] * float(record.spacing_um),
            ),
            aspect="equal",
            interpolation="nearest",
            rasterized=True,
        )
        axis.set_title(
            rf"$\epsilon={record.bulk_z_strain_percent:.2f}\%$",
            color=color,
        )
        axis.set_xlabel("transverse y (µm)")
        axis.set_ylabel("loading z (µm)")
    if image is not None:
        fig.colorbar(image, ax=list(axes.ravel()), label="leveled height (µm)")
    fig.suptitle(_case_label(*case))
    return _finish_figure(fig, "10_representative_surface_maps.png", save=save)


def plot_directional_height_profiles(
    table: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    case: tuple[int, str] = (530, "unint"),
    n_positive_states: int = 4,
    save: bool = True,
) -> plt.Figure:
    records = _representative_records(
        table, case=case, n_positive_states=n_positive_states
    )
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.5), constrained_layout=True)
    colors = _representative_colors(len(records))
    for record, color in zip(records.itertuples(index=False), colors):
        height = experimental_leveled_height(
            np.asarray(exp_heights[record.height_key], dtype=float),
            float(record.spacing_um),
        )
        z = np.arange(height.shape[0]) * float(record.spacing_um)
        y = np.arange(height.shape[1]) * float(record.spacing_um)
        label = rf"{record.bulk_z_strain_percent:.2f}%"
        axes[0].plot(z, np.mean(height, axis=1), color=color, label=label)
        axes[1].plot(y, np.mean(height, axis=0), color=color, label=label)
    axes[0].set_xlabel("loading position z (µm)")
    axes[0].set_ylabel("transverse-mean height (µm)")
    axes[0].set_title("Parallel to loading")
    axes[1].set_xlabel("transverse position y (µm)")
    axes[1].set_ylabel("loading-mean height (µm)")
    axes[1].set_title("Transverse to loading")
    axes[1].legend(title="strain")
    return _finish_figure(fig, "11_directional_height_profiles.png", save=save)


def _strain_group_palette(frame: pd.DataFrame) -> tuple[list[str], dict[str, object]]:
    order = (
        frame.groupby("strain_group", observed=True)["bulk_z_strain_percent"]
        .median()
        .sort_values()
        .index.astype(str)
        .tolist()
    )
    colors: dict[str, object] = {
        label: color for label, color in zip(order, strain_group_colors(len(order)))
    }
    return order, colors


def plot_height_distributions(
    distributions: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    """Raw and standardized map-balanced distributions using viridis groups."""
    fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.6), constrained_layout=True)
    if distributions.empty or "standardized" not in distributions:
        for axis in axes:
            axis.text(0.5, 0.5, "No positive-strain height groups", ha="center", va="center")
            axis.set_axis_off()
        return _finish_figure(fig, "12_height_distributions_by_strain.png", save=save)
    for axis, standardized in zip(axes, (False, True)):
        subset = distributions[distributions["standardized"] == standardized]
        if subset.empty:
            continue
        order, colors = _strain_group_palette(subset)
        summary = (
            subset.groupby(["strain_group", "height"], observed=True, as_index=False)[
                "density"
            ]
            .median()
        )
        for label in order:
            group = summary[summary["strain_group"].astype(str) == label]
            axis.plot(
                group["height"],
                group["density"],
                color=colors[label],
                label=label,
            )
        axis.set_xlabel(
            r"standardized height, $(h-\bar h)/S_q$"
            if standardized
            else r"leveled height, $h-\bar h$ (µm)"
        )
        axis.set_ylabel("probability density")
        axis.set_title("Standardized" if standardized else "Dimensional")
    axes[-1].legend(title="strain group")
    return _finish_figure(fig, "12_height_distributions_by_strain.png", save=save)




## PSD figures: linear wavelength, logarithmic PSD


In [ ]:
def _numeric_wavelength_axis(axis: plt.Axes) -> None:
    axis.set_xscale("linear")
    axis.set_xlim(WAVELENGTH_MIN_UM, WAVELENGTH_MAX_UM)
    axis.set_xticks(np.linspace(WAVELENGTH_MIN_UM, WAVELENGTH_MAX_UM, 7))
    axis.ticklabel_format(axis="x", style="plain", useOffset=False)
    axis.set_xlabel(r"wavelength, $\lambda$ (µm)")


def _plot_binned_curve(
    axis: plt.Axes,
    rows: pd.DataFrame,
    values: np.ndarray,
    **plot_kwargs,
) -> None:
    order = np.argsort(rows["x_lower_um"].to_numpy(dtype=float))
    ordered = rows.iloc[order]
    values = np.asarray(values, dtype=float)[order]
    valid = np.isfinite(values) & (values > 0)
    ordered = ordered.iloc[np.flatnonzero(valid)]
    values = values[valid]
    if values.size == 0:
        return
    lower = ordered["x_lower_um"].to_numpy(dtype=float)
    upper = ordered["x_upper_um"].to_numpy(dtype=float)
    if values.size > 1 and np.allclose(upper[:-1], lower[1:], rtol=1.0e-8, atol=1.0e-10):
        edges = np.concatenate(([lower[0]], upper))
        axis.stairs(values, edges, **plot_kwargs)
    else:
        axis.plot(ordered["x_um"], values, **plot_kwargs)


def _plot_normalized_psd_density(
    axis: plt.Axes,
    rows: pd.DataFrame,
    *,
    value_column: str,
    **plot_kwargs,
) -> None:
    values = rows[value_column].to_numpy(dtype=float)
    scale = _psd_density_scale(rows, value_column=value_column)
    if not np.isfinite(scale):
        return
    _plot_binned_curve(axis, rows, values * scale, **plot_kwargs)


def _psd_density_scale(
    rows: pd.DataFrame,
    *,
    value_column: str,
) -> float:
    """Scale one binned wavelength density to exact unit represented power."""
    values = rows[value_column].to_numpy(dtype=float)
    widths = (
        rows["x_upper_um"].to_numpy(dtype=float)
        - rows["x_lower_um"].to_numpy(dtype=float)
    )
    total = float(np.nansum(values * widths))
    if total <= 0 or not np.isfinite(total):
        return np.nan
    return 1.0 / total


def plot_psd_by_strain_group(
    curves: pd.DataFrame,
    *,
    case: tuple[int, str] = (530, "unint"),
    domain: str = "analysis_crop",
    save: bool = True,
) -> plt.Figure:
    """Canonical normalized PSD and absolute gain for one non-sectioned domain."""
    fig, axes = plt.subplots(1, 2, figsize=(8.7, 3.6), constrained_layout=True)
    required = {
        "source",
        "domain",
        "load_mpa",
        "sample_type",
        "strain_group",
    }
    if curves.empty or not required.issubset(curves.columns):
        for axis in axes:
            axis.text(0.5, 0.5, "No positive-strain spatial groups", ha="center", va="center")
            axis.set_axis_off()
        return _finish_figure(fig, f"13_{domain}_psd_by_strain_group.png", save=save)
    load, sample_type = case
    selected = curves[
        (curves["source"] == "exp")
        & (curves["domain"] == domain)
        & (curves["load_mpa"] == load)
        & (curves["sample_type"] == sample_type)
        & curves["strain_group"].notna()
    ].copy()
    if selected.empty:
        for axis in axes:
            axis.text(0.5, 0.5, "No positive-strain spatial groups", ha="center", va="center")
            axis.set_axis_off()
        return _finish_figure(fig, f"13_{domain}_psd_by_strain_group.png", save=save)
    order, colors = _strain_group_palette(selected)

    normalized = selected[selected["curve_type"] == "psd_normalized"]
    absolute = selected[selected["curve_type"] == "psd_absolute"]
    baseline = absolute[absolute["is_initial"]][
        ["sample_id", "x_um", "y"]
    ].rename(columns={"y": "y_initial"})
    paired_gain = absolute.merge(
        baseline,
        on=["sample_id", "x_um"],
        how="inner",
        validate="many_to_one",
    )
    paired_gain = paired_gain[
        (paired_gain["y"] > 0) & (paired_gain["y_initial"] > 0)
    ].copy()
    paired_gain["gain"] = paired_gain["y"] / paired_gain["y_initial"]
    for label in order:
        norm_group = normalized[normalized["strain_group"].astype(str) == label]
        norm_summary = (
            norm_group.groupby(["x_um", "x_lower_um", "x_upper_um"], as_index=False)[
                "y"
            ]
            .median()
        )
        _plot_normalized_psd_density(
            axes[0],
            norm_summary,
            value_column="y",
            color=colors[label],
            label=label,
        )

        raw_group = paired_gain[
            paired_gain["strain_group"].astype(str) == label
        ]
        raw_group = (
            raw_group.groupby(
                ["x_um", "x_lower_um", "x_upper_um"], as_index=False
            )["gain"]
            .median()
        )
        _plot_binned_curve(
            axes[1],
            raw_group,
            raw_group["gain"].to_numpy(dtype=float),
            color=colors[label],
            label=label,
        )
    for axis in axes:
        _numeric_wavelength_axis(axis)
        axis.set_yscale("log")
    axes[0].set_ylabel(r"normalized PSD density, $p_\lambda$ (µm$^{-1}$)")
    axes[0].set_title(f"{domain.capitalize()} height PSD")
    axes[1].set_ylabel(r"absolute PSD gain, $C(\epsilon)/C_0$")
    axes[1].set_title("Gain from unnormalized PSD")
    axes[1].axhline(1.0, color="0.55", linestyle="--", linewidth=0.8)
    axes[1].legend(title="strain group")
    return _finish_figure(fig, f"13_{domain}_psd_by_strain_group.png", save=save)


def plot_psd_band_evolution(
    table: pd.DataFrame,
    *,
    bands_um: Mapping[str, tuple[float, float]] = DEFAULT_WAVELENGTH_BANDS_UM,
    save: bool = True,
) -> plt.Figure:
    bands = validated_wavelength_bands(bands_um)
    fig, axes = plt.subplots(
        len(bands),
        2,
        figsize=(8.8, 2.8 * len(bands)),
        constrained_layout=True,
        squeeze=False,
    )
    for row_axes, (band, limits) in zip(axes, bands.items()):
        for (load, sample_type), group in table.groupby(["load_mpa", "sample_type"]):
            color = load_color(load)
            row_axes[0].scatter(
                group["bulk_z_strain_percent"],
                group[f"delta_psd_rms_{band}_um"],
                color=color,
                marker=TYPE_MARKERS[sample_type],
                s=14,
                alpha=0.65,
                label=f"{int(load)} MPa",
            )
            row_axes[1].scatter(
                group["bulk_z_strain_percent"],
                group[f"psd_gain_{band}"],
                color=color,
                marker=TYPE_MARKERS[sample_type],
                s=14,
                alpha=0.65,
            )
        title = rf"{band}: {limits[0]:.2f}–{limits[1]:.2f} µm"
        row_axes[0].set_title(title)
        row_axes[1].set_title(title)
        row_axes[0].axhline(0.0, color="0.6", linewidth=0.7)
        row_axes[1].axhline(1.0, color="0.6", linewidth=0.7, linestyle="--")
        row_axes[1].set_yscale("log")
        row_axes[0].set_ylabel(r"$\Delta\sqrt{P_{band}}$ (µm)")
        row_axes[1].set_ylabel(r"band PSD gain, $P/P_0$")
        for axis in row_axes:
            axis.set_xlabel(r"bulk axial strain, $\epsilon_{zz}$ (%)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=3)
    return _finish_figure(fig, "14_psd_band_evolution.png", save=save)




## Autocorrelation figures


In [ ]:
def plot_representative_acf_maps(
    exp_df: pd.DataFrame,
    exp_heights: Mapping[str, np.ndarray],
    *,
    domain: str = "analysis_crop",
    save: bool = True,
) -> plt.Figure:
    """One endpoint ACF map per load; maps are not additional height sections."""
    if domain != "analysis_crop":
        raise ValueError("Experimental ACF maps require analysis_crop.")
    fig, axes = _case_axes(sharex=True, sharey=True)
    image = None
    for axis, (load, sample_type) in zip(axes.ravel(), EXPERIMENTS):
        candidates = exp_df[
            (exp_df["load_mpa"] == load)
            & (exp_df["sample_type"] == sample_type)
            & exp_df["is_endpoint"]
        ]
        if candidates.empty:
            axis.text(0.5, 0.5, "not available", ha="center", va="center")
            continue
        candidate_sa = []
        for candidate in candidates.itertuples(index=False):
            candidate_height = experimental_leveled_height(
                np.asarray(exp_heights[candidate.height_key], dtype=float),
                float(candidate.spacing_um),
            )
            candidate_sa.append(surface_metrics(candidate_height)["sa_um"])
        target_sa = float(np.median(candidate_sa))
        representative_index = int(np.argmin(np.abs(np.asarray(candidate_sa) - target_sa)))
        record = candidates.iloc[representative_index]
        height = experimental_leveled_height(
            np.asarray(exp_heights[record["height_key"]], dtype=float),
            float(record["spacing_um"]),
        )
        rho = overlap_corrected_acf_2d(height)
        center = np.asarray(rho.shape) // 2
        radius_pixels = int(np.floor(ACF_MAX_LAG_UM / float(record["spacing_um"])))
        selection = (
            slice(center[0] - radius_pixels, center[0] + radius_pixels + 1),
            slice(center[1] - radius_pixels, center[1] + radius_pixels + 1),
        )
        shown = rho[selection]
        limit = radius_pixels * float(record["spacing_um"])
        image = axis.imshow(
            shown,
            origin="lower",
            extent=(-limit, limit, -limit, limit),
            cmap="coolwarm",
            vmin=-0.4,
            vmax=1.0,
            interpolation="nearest",
            rasterized=True,
        )
        axis.set_title(_case_label(load, sample_type), color=load_color(load))
        axis.set_xlabel("transverse lag (µm)")
        axis.set_ylabel("loading lag (µm)")
        axis.set_aspect("equal")
    if image is not None:
        fig.colorbar(image, ax=list(axes.ravel()), label="normalized linear ACF")
    return _finish_figure(fig, f"15_{domain}_representative_acf_maps.png", save=save)


def plot_acf_by_strain_group(
    curves: pd.DataFrame,
    *,
    case: tuple[int, str] = (530, "unint"),
    domain: str = "analysis_crop",
    save: bool = True,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6.3, 4.0), constrained_layout=True)
    required = {
        "source",
        "domain",
        "load_mpa",
        "sample_type",
        "curve_type",
        "strain_group",
    }
    if curves.empty or not required.issubset(curves.columns):
        ax.text(0.5, 0.5, "No positive-strain spatial groups", ha="center", va="center")
        ax.set_axis_off()
        return _finish_figure(fig, f"16_{domain}_acf_by_strain_group.png", save=save)
    load, sample_type = case
    selected = curves[
        (curves["source"] == "exp")
        & (curves["domain"] == domain)
        & (curves["load_mpa"] == load)
        & (curves["sample_type"] == sample_type)
        & (curves["curve_type"] == "acf")
        & curves["strain_group"].notna()
    ]
    if selected.empty:
        ax.text(0.5, 0.5, "No positive-strain spatial groups", ha="center", va="center")
        ax.set_axis_off()
        return _finish_figure(fig, f"16_{domain}_acf_by_strain_group.png", save=save)
    order, colors = _strain_group_palette(selected)
    for label in order:
        group = selected[selected["strain_group"].astype(str) == label]
        summary = group.groupby("x_um", as_index=False)["y"].median()
        ax.plot(summary["x_um"], summary["y"], color=colors[label], label=label)
    ax.axhline(np.exp(-1.0), color="0.55", linestyle=":", linewidth=0.8)
    ax.axhline(0.0, color="0.65", linewidth=0.7)
    ax.set_xlim(0.0, ACF_MAX_LAG_UM)
    ax.set_xlabel("radial lag (µm)")
    ax.set_ylabel(r"radial ACF, $\rho(r)$")
    ax.legend(title="strain group")
    return _finish_figure(fig, f"16_{domain}_acf_by_strain_group.png", save=save)


def plot_acf_length_vs_strain(
    scalars: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6.4, 4.0), constrained_layout=True)
    required = {
        "load_mpa",
        "sample_type",
        "bulk_z_strain_percent",
        "acf_one_over_e_um",
        "acf_one_over_e_crossed",
    }
    if scalars.empty or not required.issubset(scalars.columns):
        ax.text(0.5, 0.5, "No ACF-length observations", ha="center", va="center")
        ax.set_axis_off()
        return _finish_figure(fig, "17_acf_length_vs_strain.png", save=save)
    for (load, sample_type), group in scalars.groupby(["load_mpa", "sample_type"]):
        measured = group[group["acf_one_over_e_crossed"]]
        censored = group[~group["acf_one_over_e_crossed"]]
        ax.scatter(
            measured["bulk_z_strain_percent"],
            measured["acf_one_over_e_um"],
            color=load_color(load),
            marker=TYPE_MARKERS[sample_type],
            label=f"{int(load)} MPa",
            s=18,
        )
        if not censored.empty:
            ax.scatter(
                censored["bulk_z_strain_percent"],
                np.full(len(censored), ACF_MAX_LAG_UM),
                facecolor="none",
                edgecolor=load_color(load),
                marker="v",
                s=22,
            )
    ax.set_xlabel(r"bulk axial strain, $\epsilon_{zz}$ (%)")
    ax.set_ylabel(r"first $1/e$ ACF length (µm)")
    ax.legend(ncol=2)
    return _finish_figure(fig, "17_acf_length_vs_strain.png", save=save)




## Manuscript Figure 9 and Figure 10 reproductions


In [ ]:
def plot_paired_spatial_evolution(
    paired_curves: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6), constrained_layout=True)
    if paired_curves.empty:
        for axis in axes:
            axis.text(0.5, 0.5, "No eligible paired specimens", ha="center", va="center")
            axis.set_axis_off()
        return _finish_figure(fig, "18_manuscript_figure9_paired_evolution.png", save=save)
    summary = summarize_paired_curves(paired_curves)
    for load in (500, 530):
        color = load_color(load)
        gain = summary[
            (summary["load_mpa"] == load)
            & (summary["curve_type"] == "psd_gain_decades")
        ].sort_values("x_um")
        axes[0].plot(gain["x_um"], gain["median"], color=color, label=f"{load} MPa")
        axes[0].fill_between(
            gain["x_um"], gain["q25"], gain["q75"], color=color, alpha=0.2
        )
        acf = summary[
            (summary["load_mpa"] == load)
            & (summary["curve_type"] == "acf_change")
        ].sort_values("x_um")
        axes[1].plot(acf["x_um"], acf["median"], color=color, label=f"{load} MPa")
        axes[1].fill_between(
            acf["x_um"], acf["q25"], acf["q75"], color=color, alpha=0.2
        )
    _numeric_wavelength_axis(axes[0])
    axes[0].axhline(0.0, color="0.6", linewidth=0.7)
    axes[0].set_ylabel(r"paired log$_{10}$ absolute PSD gain")
    axes[1].set_xlim(0.0, ACF_MAX_LAG_UM)
    axes[1].axhline(0.0, color="0.6", linewidth=0.7)
    axes[1].set_xlabel("radial lag (µm)")
    axes[1].set_ylabel(r"paired ACF change, $\Delta\rho(r)$")
    axes[1].legend()
    return _finish_figure(fig, "18_manuscript_figure9_paired_evolution.png", save=save)


def plot_endpoint_spatial_comparison(
    summary: pd.DataFrame,
    *,
    psd_curve_type: str = "psd_normalized",
    filename: str = "19_manuscript_figure10_endpoint_spatial_comparison.png",
    title_prefix: str = "Matched 256×128 µm operator",
    save: bool = True,
) -> plt.Figure:
    """Endpoint experiment/simulation comparison with declared PSD definition."""
    allowed = {"psd_normalized", "psd_legacy_diagnostic"}
    if psd_curve_type not in allowed:
        raise ValueError(f"psd_curve_type must be one of {sorted(allowed)}.")
    fig, axes = plt.subplots(2, 2, figsize=(9.0, 6.8), constrained_layout=True)
    for row_index, (sample_type, loads) in enumerate(
        (("int", (475, 525, 575)), ("unint", (500, 530, 588)))
    ):
        psd_axis, acf_axis = axes[row_index]
        for load in loads:
            for source in ("exp", "sim"):
                psd = summary[
                    (summary["source"] == source)
                    & (summary["load_mpa"] == load)
                    & (summary["sample_type"] == sample_type)
                    & (summary["curve_type"] == psd_curve_type)
                ].copy()
                if psd.empty:
                    continue
                statistic = "median" if source == "exp" else "mean"
                label = f"{load} {source}"
                if load == 588 and source == "exp":
                    label += " (descriptive)"
                kwargs = {
                    "color": load_color(load),
                    "linestyle": SOURCE_LINESTYLES[source],
                    "label": label,
                }
                density_scale = 1.0
                if psd_curve_type == "psd_normalized":
                    density_scale = _psd_density_scale(
                        psd,
                        value_column=statistic,
                    )
                    _plot_normalized_psd_density(
                        psd_axis, psd, value_column=statistic, **kwargs
                    )
                else:
                    _plot_binned_curve(
                        psd_axis,
                        psd,
                        psd[statistic].to_numpy(dtype=float),
                        **kwargs,
                    )
                if source == "exp" and {"q25", "q75"}.issubset(psd.columns):
                    psd_sorted = psd.sort_values("x_um")
                    psd_axis.fill_between(
                        psd_sorted["x_um"],
                        psd_sorted["q25"] * density_scale,
                        psd_sorted["q75"] * density_scale,
                        color=load_color(load),
                        alpha=0.10,
                        linewidth=0,
                    )

                acf = summary[
                    (summary["source"] == source)
                    & (summary["load_mpa"] == load)
                    & (summary["sample_type"] == sample_type)
                    & (summary["curve_type"] == "acf")
                ].sort_values("x_um")
                if not acf.empty:
                    acf_axis.plot(
                        acf["x_um"],
                        acf[statistic],
                        color=load_color(load),
                        linestyle=SOURCE_LINESTYLES[source],
                        label=label,
                    )
                    if source == "exp" and {"q25", "q75"}.issubset(acf.columns):
                        acf_axis.fill_between(
                            acf["x_um"],
                            acf["q25"],
                            acf["q75"],
                            color=load_color(load),
                            alpha=0.10,
                            linewidth=0,
                        )
        _numeric_wavelength_axis(psd_axis)
        psd_axis.set_yscale("log")
        psd_axis.set_ylabel(
            r"normalized PSD density, $p_\lambda$ (µm$^{-1}$)"
            if psd_curve_type == "psd_normalized"
            else "legacy frequency-area-normalized radial PSD"
        )
        psd_axis.set_title(
            "Interrupted" if sample_type == "int" else "Uninterrupted"
        )
        acf_axis.set_xlim(0.0, ACF_MAX_LAG_UM)
        acf_axis.set_ylim(-0.35, 1.05)
        acf_axis.axhline(np.exp(-1.0), color="0.55", linestyle=":", linewidth=0.8)
        acf_axis.axhline(0.0, color="0.65", linewidth=0.7)
        acf_axis.set_xlabel("radial lag (µm)")
        acf_axis.set_ylabel(r"radial ACF, $\rho(r)$")
        acf_axis.legend(ncol=2)
    fig.suptitle(title_prefix)
    return _finish_figure(fig, filename, save=save)


def plot_legacy_manuscript_psd_diagnostic(
    summary_with_legacy: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    """Forensic-only renderer for the manuscript's older frequency-area curve."""
    return plot_endpoint_spatial_comparison(
        summary_with_legacy,
        psd_curve_type="psd_legacy_diagnostic",
        filename="19b_legacy_manuscript_psd_normalization_diagnostic.png",
        title_prefix="Diagnostic only: legacy frequency-area normalization",
        save=save,
    )


def plot_nonsectioned_paired_evolution(
    changes: pd.DataFrame,
    *,
    domain: str = "analysis_crop",
    save: bool = True,
) -> plt.Figure:
    """Preserved whole-analysis-crop paired change, explicitly not sectioned."""
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.6), constrained_layout=True)
    if changes.empty or "domain" not in changes:
        for axis in axes:
            axis.text(0.5, 0.5, "No eligible paired specimens", ha="center", va="center")
            axis.set_axis_off()
        return _finish_figure(
            fig, "20_nonsectioned_analysis_crop_paired_evolution.png", save=save
        )
    selected = changes[changes["domain"] == domain]
    for (load, sample_type), case in selected.groupby(["load_mpa", "sample_type"]):
        color = load_color(load)
        gain = case[case["change_type"] == "psd_gain_decades"]
        gain_summary = gain.groupby("x_um", as_index=False)["change"].median()
        axes[0].plot(
            gain_summary["x_um"],
            gain_summary["change"],
            color=color,
            label=f"{int(load)} MPa",
        )
        acf = case[case["change_type"] == "acf_change"]
        acf_summary = acf.groupby("x_um", as_index=False)["change"].median()
        axes[1].plot(
            acf_summary["x_um"],
            acf_summary["change"],
            color=color,
            label=f"{int(load)} MPa",
        )
    _numeric_wavelength_axis(axes[0])
    axes[0].axhline(0.0, color="0.6", linewidth=0.7)
    axes[0].set_ylabel(r"paired log$_{10}$ absolute PSD gain")
    axes[1].set_xlim(0.0, ACF_MAX_LAG_UM)
    axes[1].axhline(0.0, color="0.6", linewidth=0.7)
    axes[1].set_xlabel("radial lag (µm)")
    axes[1].set_ylabel(r"paired ACF change, $\Delta\rho(r)$")
    axes[1].legend(ncol=2)
    return _finish_figure(
        fig, "20_nonsectioned_analysis_crop_paired_evolution.png", save=save
    )


def plot_delta_h_psd_acf_sensitivity(
    initial_raw_height_um: np.ndarray,
    final_raw_height_um: np.ndarray,
    spacing_um: float,
    *,
    save: bool = True,
) -> plt.Figure:
    """Signed, unregistered crop-first Delta-h sensitivity retained explicitly."""
    delta = delta_h_unregistered_sensitivity(
        initial_raw_height_um,
        final_raw_height_um,
        spacing_um,
    )
    spatial = nonsectioned_spatial_curves(delta, spacing_um)
    fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4), constrained_layout=True)
    limit = float(np.percentile(np.abs(delta), 99.0))
    image = axes[0].imshow(
        delta,
        origin="lower",
        cmap="coolwarm",
        vmin=-limit,
        vmax=limit,
        extent=(0, delta.shape[1] * spacing_um, 0, delta.shape[0] * spacing_um),
        aspect="equal",
        interpolation="nearest",
        rasterized=True,
    )
    fig.colorbar(image, ax=axes[0], label=r"signed $\Delta h$ (µm)")
    axes[0].set_xlabel("transverse y (µm)")
    axes[0].set_ylabel("loading z (µm)")
    axes[0].set_title("Unregistered analysis-crop difference")

    psd_rows = pd.DataFrame(
        {
            "x_um": spatial["wavelength_um"],
            "x_lower_um": spatial["wavelength_lower_um"],
            "x_upper_um": spatial["wavelength_upper_um"],
            "density": spatial["normalized_psd_um_inv"],
        }
    ).dropna()
    _plot_normalized_psd_density(
        axes[1], psd_rows, value_column="density", color="black"
    )
    _numeric_wavelength_axis(axes[1])
    axes[1].set_yscale("log")
    axes[1].set_ylabel(r"normalized $\Delta h$ PSD density (µm$^{-1}$)")
    axes[1].set_title("Scale distribution")

    axes[2].plot(spatial["acf_lag_um"], spatial["acf"], color="black")
    axes[2].axhline(np.exp(-1.0), color="0.55", linestyle=":", linewidth=0.8)
    axes[2].axhline(0.0, color="0.65", linewidth=0.7)
    axes[2].set_xlim(0.0, ACF_MAX_LAG_UM)
    axes[2].set_xlabel("radial lag (µm)")
    axes[2].set_ylabel(r"$\Delta h$ ACF")
    axes[2].set_title(
        rf"$1/e$={spatial['acf_one_over_e_um']:.2f} µm"
    )
    return _finish_figure(fig, "21_unregistered_delta_h_sensitivity.png", save=save)


def plot_endpoint_spatial_descriptors(
    scalars: pd.DataFrame,
    *,
    save: bool = True,
) -> plt.Figure:
    """Endpoint spectral median, long-power fraction, and ACF length."""
    specifications = (
        ("spectral_median_wavelength_um", "spectral-median wavelength (µm)"),
        ("long_wavelength_power_fraction", r"power fraction, $\lambda\geq32$ µm"),
        ("acf_one_over_e_um", r"first $1/e$ ACF length (µm)"),
    )
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.5), constrained_layout=True)
    for axis, (column, ylabel) in zip(axes, specifications):
        for source, source_group in scalars.groupby("source"):
            if source == "sim":
                source_group = (
                    source_group.groupby(
                        ["load_mpa", "sample_type", "micro_id"], as_index=False
                    )[column]
                    .mean()
                )
            for (load, sample_type), group in source_group.groupby(
                ["load_mpa", "sample_type"]
            ):
                values = group[column].dropna()
                if values.empty:
                    continue
                axis.errorbar(
                    load + (-2.0 if source == "exp" else 2.0),
                    values.mean(),
                    yerr=values.std(ddof=1) if len(values) > 1 else None,
                    color=load_color(load),
                    marker=TYPE_MARKERS[sample_type] if source == "exp" else "D",
                    markerfacecolor=load_color(load) if source == "exp" else "none",
                    linestyle="none",
                    capsize=2,
                )
        axis.set_xlabel("applied stress (MPa)")
        axis.set_ylabel(ylabel)
    return _finish_figure(fig, "22_endpoint_spatial_descriptors.png", save=save)


def plot_psd_estimator_diagnostic(
    raw_height_um: np.ndarray,
    spacing_um: float,
    *,
    save: bool = True,
) -> plt.Figure:
    """One retained 2D periodogram/Parseval diagnostic on the analysis crop."""
    height = experimental_leveled_height(raw_height_um, spacing_um)
    f0, f1, psd2d, error = periodogram_2d(height, spacing_um)
    shifted = np.fft.fftshift(psd2d)
    fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.4), constrained_layout=True)
    axes[0].imshow(
        height,
        origin="lower",
        cmap="coolwarm",
        extent=(0, height.shape[1] * spacing_um, 0, height.shape[0] * spacing_um),
        aspect="equal",
        interpolation="nearest",
        rasterized=True,
    )
    axes[0].set_title("Leveled analysis crop")
    axes[0].set_xlabel("transverse y (µm)")
    axes[0].set_ylabel("loading z (µm)")
    positive = shifted[shifted > 0]
    norm = mpl.colors.LogNorm(vmin=np.percentile(positive, 2), vmax=np.percentile(positive, 99.8))
    image = axes[1].imshow(
        shifted,
        origin="lower",
        cmap="viridis",
        norm=norm,
        extent=(np.fft.fftshift(f1)[0], np.fft.fftshift(f1)[-1], np.fft.fftshift(f0)[0], np.fft.fftshift(f0)[-1]),
        aspect="equal",
        interpolation="nearest",
        rasterized=True,
    )
    fig.colorbar(image, ax=axes[1], label=r"2D PSD (µm$^4$)")
    axes[1].set_xlabel(r"$f_y$ (µm$^{-1}$)")
    axes[1].set_ylabel(r"$f_z$ (µm$^{-1}$)")
    axes[1].set_title(f"Parseval relative error={error:.2e}")
    return _finish_figure(fig, "23_psd_estimator_diagnostic.png", save=save)




## Configuration audit and all-figure orchestration


In [ ]:
def analysis_configuration_report() -> dict[str, object]:
    """Machine-readable geometry/bandwidth statement for this file."""
    crop_shape = tuple(
        len(range(*crop_slice.indices(size)))
        for crop_slice, size in zip(EXP_ANALYSIS_CROP, EXPECTED_EXP_SHAPE)
    )
    resampled_shape = tuple(
        int(np.floor((size - 1) * EXP_NATIVE_SPACING_UM / TARGET_SPACING_UM)) + 1
        for size in crop_shape
    )
    window_shape = tuple(
        int(round(length / TARGET_SPACING_UM))
        for length in SPATIAL_WINDOW_SHAPE_UM
    )
    window_grid = tuple(
        size // window for size, window in zip(resampled_shape, window_shape)
    )
    return {
        "experimental_raw_shape": list(EXPECTED_EXP_SHAPE),
        "experimental_analysis_crop": ["50:-50", "50:750"],
        "experimental_crop_shape": list(crop_shape),
        "native_10x_spacing_um": EXP_NATIVE_SPACING_UM,
        "native_10x_nyquist_um_inv": EXP_NATIVE_NYQUIST_UM_INV,
        "analysis_grid_spacing_um": TARGET_SPACING_UM,
        "analysis_grid_is_interpolated_not_resolution": True,
        "resampled_crop_center_grid_shape": list(resampled_shape),
        "section_window_shape_um": list(SPATIAL_WINDOW_SHAPE_UM),
        "section_window_grid": list(window_grid),
        "sections_per_experimental_map": int(np.prod(window_grid)),
        "wavelength_min_um": WAVELENGTH_MIN_UM,
        "wavelength_max_um": WAVELENGTH_MAX_UM,
        "acf_max_lag_um": ACF_MAX_LAG_UM,
        "matched_annuli": N_MATCHED_FREQUENCY_ANNULI,
        "psd_display_normalization": (
            "exact retained 2D mode power / total retained power / wavelength-bin width"
        ),
        "legacy_normalization_status": "diagnostic only",
    }


def validate_analysis_configuration() -> dict[str, object]:
    report = analysis_configuration_report()
    if report["sections_per_experimental_map"] != 21:
        raise AssertionError(
            "Crop-first geometry no longer yields the expected 21 manuscript-footprint windows."
        )
    if matched_frequency_edges()[-1] > EXP_NATIVE_NYQUIST_UM_INV + 1.0e-12:
        raise AssertionError("Matched PSD exceeds the native experimental Nyquist limit.")
    if TARGET_SPACING_UM < EXP_NATIVE_SPACING_UM and not report[
        "analysis_grid_is_interpolated_not_resolution"
    ]:
        raise AssertionError("Upsampled grid must not be described as optical resolution.")
    return report


def _save_table(frame: pd.DataFrame, name: str) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frame.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)


def run_all_figures(
    *,
    rebuild_caches: bool = False,
    include_expensive_spatial: bool = True,
    include_legacy_diagnostic: bool = False,
    dic_csv_path: str | Path | None = DEFAULT_DIC_CSV_PATH,
    save: bool = True,
    show: bool = False,
) -> dict[str, object]:
    """Build inputs once, make grouped figures, save derived tables, and return all products."""
    configuration = validate_analysis_configuration()
    cached_exp_df, cached_sim_df, cached_exp_heights, cached_sim_heights = (
        initialize_caches(rebuild=rebuild_caches)
    )

    exp_roughness = experimental_roughness_table(cached_exp_df, cached_exp_heights)
    sim_roughness = simulation_roughness_table(cached_sim_df, cached_sim_heights)
    grouped_exp, _, _ = assign_strain_groups(exp_roughness, n_groups=4)
    histories = mechanical_history_table(cached_exp_df, cached_sim_df)
    rates = incremental_roughening_rates(exp_roughness)
    model_fits = fit_roughening_models(exp_roughness)
    band_table = experimental_psd_band_table(
        exp_roughness,
        cached_exp_heights,
        domain="analysis_crop",
    )
    dimensional_distributions = height_distribution_curves(
        grouped_exp,
        cached_exp_heights,
        domain="analysis_crop",
        standardized=False,
    )
    standardized_distributions = height_distribution_curves(
        grouped_exp,
        cached_exp_heights,
        domain="analysis_crop",
        standardized=True,
    )
    distributions = pd.concat(
        [dimensional_distributions, standardized_distributions], ignore_index=True
    )

    tables: dict[str, pd.DataFrame] = {
        "experimental_roughness": exp_roughness,
        "simulation_roughness": sim_roughness,
        "mechanical_histories": histories,
        "incremental_roughening_rates": rates,
        "roughening_model_fits": model_fits,
        "psd_band_metrics": band_table,
        "height_distributions": distributions,
    }
    figures: list[plt.Figure] = []

    if CALIBRATION_PARAMS_PATH.exists():
        calibration = calibration_parameter_table(CALIBRATION_PARAMS_PATH)
        tables["calibration_parameters"] = calibration
        figures.append(plot_calibration_parameters(calibration, save=save))
    figures.append(plot_mechanical_strain_histories(histories, save=save))

    representative_case = _choose_representative_case(exp_roughness)
    representative = _representative_records(
        exp_roughness,
        case=representative_case,
    )
    representative_record = representative.iloc[-1]
    representative_raw = np.asarray(
        cached_exp_heights[representative_record["height_key"]], dtype=float
    )
    figures.append(
        plot_raw_map_qc_and_analysis_crop(
            representative_raw,
            float(representative_record["spacing_um"]),
            title=(
                f"{representative_record['load_mpa']} MPa, "
                f"sample {representative_record['sample_id']}"
            ),
            save=save,
        )
    )
    figures.append(
        plot_roughness_case_panels(
            exp_roughness, sim_roughness, x_column="time_h", save=save
        )
    )
    figures.append(
        plot_roughness_case_panels(
            exp_roughness,
            sim_roughness,
            x_column="bulk_z_strain_percent",
            save=save,
        )
    )
    figures.append(plot_endpoint_delta_sa_by_load(exp_roughness, sim_roughness, save=save))
    figures.append(plot_incremental_roughening_rates(rates, save=save))
    if not model_fits.empty:
        figures.append(
            plot_roughening_model_comparison(
                exp_roughness,
                model_fits,
                case=representative_case,
                save=save,
            )
        )
    figures.append(plot_surface_parameter_suite(exp_roughness, save=save))
    figures.append(
        plot_representative_surface_maps(
            exp_roughness,
            cached_exp_heights,
            case=representative_case,
            save=save,
        )
    )
    figures.append(
        plot_directional_height_profiles(
            exp_roughness,
            cached_exp_heights,
            case=representative_case,
            save=save,
        )
    )
    figures.append(plot_height_distributions(distributions, save=save))
    figures.append(plot_psd_band_evolution(band_table, save=save))

    if include_expensive_spatial:
        exp_nonsectioned_curves, exp_nonsectioned_scalars = (
            nonsectioned_spatial_tables(
                grouped_exp,
                cached_exp_heights,
                source="exp",
                domains=("analysis_crop",),
            )
        )
        sim_endpoint_frame = cached_sim_df[
            cached_sim_df["is_endpoint"]
            & cached_sim_df["unique_plane"]
            & cached_sim_df["micro_id"].isin(PRIMARY_MORPHOLOGY_MICRO_IDS)
        ]
        sim_nonsectioned_curves, sim_nonsectioned_scalars = (
            nonsectioned_spatial_tables(
                sim_endpoint_frame,
                cached_sim_heights,
                source="sim",
                domains=("full_face",),
            )
        )
        nonsectioned_curves = pd.concat(
            [exp_nonsectioned_curves, sim_nonsectioned_curves], ignore_index=True
        )
        nonsectioned_scalars = pd.concat(
            [exp_nonsectioned_scalars, sim_nonsectioned_scalars], ignore_index=True
        )
        tables["nonsectioned_spatial_curves"] = nonsectioned_curves
        tables["nonsectioned_spatial_scalars"] = nonsectioned_scalars

        figures.append(
            plot_psd_by_strain_group(
                exp_nonsectioned_curves,
                case=representative_case,
                domain="analysis_crop",
                save=save,
            )
        )
        figures.append(
            plot_representative_acf_maps(
                cached_exp_df, cached_exp_heights, save=save
            )
        )
        figures.append(
            plot_acf_by_strain_group(
                exp_nonsectioned_curves,
                case=representative_case,
                domain="analysis_crop",
                save=save,
            )
        )
        figures.append(plot_acf_length_vs_strain(exp_nonsectioned_scalars, save=save))
        whole_crop_changes = (
            paired_nonsectioned_spatial_evolution(exp_nonsectioned_curves)
            if not exp_nonsectioned_curves.empty
            else pd.DataFrame()
        )
        tables["nonsectioned_paired_changes"] = whole_crop_changes
        figures.append(
            plot_nonsectioned_paired_evolution(
                whole_crop_changes, domain="analysis_crop", save=save
            )
        )
        nonsectioned_endpoint_summary = summarize_nonsectioned_endpoint_curves(
            nonsectioned_curves
        )
        tables["nonsectioned_endpoint_summary"] = nonsectioned_endpoint_summary
        figures.append(
            plot_endpoint_spatial_comparison(
                nonsectioned_endpoint_summary,
                filename="20b_nonsectioned_crop_vs_face_endpoint_comparison.png",
                title_prefix="Non-sectioned experimental analysis crop vs simulation face",
                save=save,
            )
        )

        endpoint_curves, endpoint_scalars = endpoint_spatial_curves(
            cached_exp_df,
            cached_sim_df,
            cached_exp_heights,
            cached_sim_heights,
            include_588_descriptive=True,
            include_legacy_diagnostic=include_legacy_diagnostic,
        )
        endpoint_summary = summarize_endpoint_curves(endpoint_curves)
        tables["matched_endpoint_curves"] = endpoint_curves
        tables["matched_endpoint_scalars"] = endpoint_scalars
        tables["matched_endpoint_summary"] = endpoint_summary
        tables["matched_endpoint_scalar_summary"] = summarize_endpoint_scalars(
            endpoint_scalars
        )
        figures.append(plot_endpoint_spatial_comparison(endpoint_summary, save=save))
        figures.append(plot_endpoint_spatial_descriptors(endpoint_scalars, save=save))
        if include_legacy_diagnostic:
            figures.append(
                plot_legacy_manuscript_psd_diagnostic(endpoint_summary, save=save)
            )

        paired_curves, paired_scalars = paired_spatial_evolution(
            cached_exp_df, cached_exp_heights
        )
        tables["matched_paired_curves"] = paired_curves
        tables["matched_paired_scalars"] = paired_scalars
        figures.append(plot_paired_spatial_evolution(paired_curves, save=save))

        if len(representative) >= 2:
            initial_record = representative.iloc[0]
            final_record = representative.iloc[-1]
            figures.append(
                plot_delta_h_psd_acf_sensitivity(
                    np.asarray(
                        cached_exp_heights[initial_record["height_key"]], dtype=float
                    ),
                    np.asarray(
                        cached_exp_heights[final_record["height_key"]], dtype=float
                    ),
                    float(initial_record["spacing_um"]),
                    save=save,
                )
            )
        figures.append(
            plot_psd_estimator_diagnostic(
                representative_raw,
                float(representative_record["spacing_um"]),
                save=save,
            )
        )

    if dic_csv_path is not None:
        dic_datasets = read_dic_blocks(dic_csv_path)
        figures.extend(plot_dic_final_strain_fields(dic_datasets, save=save))
        spectral_tables = []
        for dataset_index, (dataset, crop) in enumerate(zip(dic_datasets, DIC_CROPS)):
            fields = dic_strain_fields(dataset, crop)
            spectrum = dic_axial_row_averaged_psd(fields)
            spectrum["dataset_index"] = dataset_index
            spectral_tables.append(spectrum)
        tables["dic_axial_row_averaged_psd"] = pd.concat(
            spectral_tables, ignore_index=True
        )

    if save:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        (OUTPUT_DIR / "analysis_configuration.json").write_text(
            json.dumps(configuration, indent=2, sort_keys=True) + "\n",
            encoding="utf-8",
        )
        for name, frame in tables.items():
            _save_table(frame, name)
    if show:
        plt.show()
    else:
        for figure in figures:
            plt.close(figure)
    return {
        "configuration": configuration,
        "exp_df": cached_exp_df,
        "sim_df": cached_sim_df,
        "exp_heights": cached_exp_heights,
        "sim_heights": cached_sim_heights,
        "tables": tables,
        "figures": figures,
    }


## Run the complete figure pipeline

Set `RUN_ALL_FIGURES` to `True` only after the four caches and configured source paths are available.


In [ ]:
RUN_ALL_FIGURES = False

if RUN_ALL_FIGURES:
    products = run_all_figures(
        rebuild_caches=False,
        include_expensive_spatial=True,
        include_legacy_diagnostic=False,
        save=True,
        show=False,
    )
